# Meta-Locked Utah Transfer Learning — task-only adapter

This notebook preserves one-second physical timing, uses the correct Utah-to-Meta output mapping, keeps the Meta backbone completely frozen, and aligns labels using the official model's exact left-context and stride.

Training uses only the mapped active-valid task BCE from the Utah trials. Only the pretrained checkpoint and Utah dataset are required.

Run Cells 1–4 first. Cell 4 is a mandatory non-training structural diagnostic; Cell 5 trains only when those checks pass.

Cell 2 intentionally excludes only training trials whose final saved labels contain no active-valid EMG burst. Validation and test trials are never silently excluded. The main reported metrics are training/validation task BCE and complete-trial mean-logit accuracy. Majority-vote accuracy is not computed or displayed.


In [1]:
# ============================================================
# CELL 1 — CONFIGURATION + FROZEN META MODEL + 2 kHz ADAPTER
# ============================================================

from pathlib import Path
import sys
import random
import copy

import numpy as np
import torch
from torch import nn

try:
    from scipy.signal import butter, sosfiltfilt, resample_poly
except ImportError as exc:
    raise ImportError(
        "This notebook requires scipy for anti-aliased resampling and "
        "40 Hz high-pass filtering. Install it with: pip install scipy"
    ) from exc

try:
    from torch.nn.utils.parametrizations import weight_norm
except ImportError:
    from torch.nn.utils import weight_norm


# ------------------------------------------------------------
# Paths
# ------------------------------------------------------------

REPO_ROOT = Path(r"C:\Users\Micah\utah-neuro\generic_neuromotor_interface")
MODEL_DIR = REPO_ROOT / "emg_models" / "discrete_gestures"
CKPT_PATH = MODEL_DIR / "model_checkpoint.ckpt"

BASE_DIR = REPO_ROOT
DATA_PATH = BASE_DIR / "Gesture_Trial_Dataset_Labeled.pt"

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from generic_neuromotor_interface.networks import DiscreteGesturesArchitecture


# ------------------------------------------------------------
# Reproducibility
# ------------------------------------------------------------

SEED = 0

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True


# ------------------------------------------------------------
# Data/model timing and output semantics
# ------------------------------------------------------------

YOUR_CHANNELS = 32
META_CHANNELS = 16

YOUR_SAMPLING_RATE = 30_000
META_SAMPLING_RATE = 2_000

RAW_INPUT_SAMPLES = 30_000       # one second at 30 kHz
META_INPUT_SAMPLES = 2_000       # one second at 2 kHz
UTAH_TO_META_DOWNSAMPLE = YOUR_SAMPLING_RATE // META_SAMPLING_RATE

if YOUR_SAMPLING_RATE % META_SAMPLING_RATE != 0:
    raise ValueError("Utah-to-Meta sampling-rate ratio must be an integer.")

NUM_META_CLASSES = 9
AO_KEPT_CLASSES = 5

AO_CLASS_NAMES = [
    "thumb left",
    "thumb right",
    "thumb up",
    "thumb down",
    "thumb press",
]

# Meta output order:
# 0 index press
# 1 index release
# 2 middle press
# 3 middle release
# 4 thumb tap
# 5 thumb swipe left
# 6 thumb swipe right
# 7 thumb swipe up
# 8 thumb swipe down
UTAH_TO_META_OUTPUTS = [5, 6, 7, 8, 4]

if len(UTAH_TO_META_OUTPUTS) != AO_KEPT_CLASSES:
    raise ValueError("UTAH_TO_META_OUTPUTS must contain five indices.")

if len(set(UTAH_TO_META_OUTPUTS)) != AO_KEPT_CLASSES:
    raise ValueError("UTAH_TO_META_OUTPUTS contains duplicate indices.")

if not all(0 <= index < NUM_META_CLASSES for index in UTAH_TO_META_OUTPUTS):
    raise ValueError("UTAH_TO_META_OUTPUTS contains an invalid Meta output index.")


# ------------------------------------------------------------
# Training configuration
# ------------------------------------------------------------

BATCH_SIZE = 16
AO_EPOCHS = 200

AO_LR = 0.01
AO_WEIGHT_DECAY = 1e-3
AO_GRAD_CLIP_NORM = 1.0

AO_WARMUP_EPOCHS = 5
AO_DECAY_EPOCH = 26
AO_DECAY_FACTOR = 0.5

# The saved Utah dataset is expected to already contain the previously chosen
# 100 ms forward label shift. This notebook does NOT shift labels a second time.
UTAH_LABELS_ALREADY_SHIFTED_100_MS = True

EXPECTED_SPLIT_SIZES = {
    "train": 80,
    "val": 10,
    "test": 10,
}


# ------------------------------------------------------------
# Preprocessing configuration
# ------------------------------------------------------------

HIGH_PASS_HZ = 40.0
HIGH_PASS_ORDER = 4

# Utah hardware units are dataset-specific. Change only if independently
# justified for this dataset.
UTAH_INPUT_SCALE = 1.0

# Official DiscreteGesturesArchitecture returns logits, not probabilities.
META_OUTPUT_IS_PROBABILITY = False


# ------------------------------------------------------------
# Adapter configuration
# ------------------------------------------------------------

ADAPTER_HIDDEN_CHANNELS = 48
ADAPTER_GROUPS = 8
ADAPTER_KERNEL_SIZE = 5
INITIAL_RESIDUAL_SCALE = 0.10
INITIAL_OUTPUT_GAIN = 0.25

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")


# ------------------------------------------------------------
# Signal preprocessing helpers
# ------------------------------------------------------------

def make_highpass_sos(sample_rate, cutoff_hz=40.0, order=4):
    if not 0 < cutoff_hz < sample_rate / 2:
        raise ValueError(
            f"High-pass cutoff {cutoff_hz} Hz is invalid for "
            f"sample rate {sample_rate} Hz."
        )

    return butter(
        order,
        cutoff_hz,
        btype="highpass",
        fs=sample_rate,
        output="sos",
    )


META_HIGHPASS_SOS = make_highpass_sos(
    META_SAMPLING_RATE,
    HIGH_PASS_HZ,
    HIGH_PASS_ORDER,
)


def highpass_numpy_channels_time(x_ct, sos=META_HIGHPASS_SOS):
    """
    Apply a zero-phase 40 Hz high-pass to [channels,time].
    """
    x = np.asarray(x_ct, dtype=np.float32)

    if x.ndim != 2:
        raise ValueError(f"Expected [channels,time], got {x.shape}")

    if x.shape[1] < 32:
        raise ValueError(
            f"Signal is too short for stable high-pass filtering: {x.shape}"
        )

    return sosfiltfilt(sos, x, axis=1).astype(np.float32, copy=False)


def resample_utah_30k_to_2k(x_ct):
    """
    Anti-aliased rational resampling:
        [32,30000] at 30 kHz -> [32,2000] at 2 kHz.
    """
    x = np.asarray(x_ct, dtype=np.float32)

    if x.ndim != 2:
        raise ValueError(f"Expected [channels,time], got {x.shape}")

    if x.shape[0] != YOUR_CHANNELS:
        raise ValueError(
            f"Expected {YOUR_CHANNELS} Utah channels, got {x.shape}"
        )

    if x.shape[1] != RAW_INPUT_SAMPLES:
        raise ValueError(
            f"Expected exactly {RAW_INPUT_SAMPLES} Utah samples, got {x.shape}"
        )

    y = resample_poly(
        x,
        up=1,
        down=UTAH_TO_META_DOWNSAMPLE,
        axis=1,
    )

    if y.shape != (YOUR_CHANNELS, META_INPUT_SAMPLES):
        raise RuntimeError(
            "Unexpected anti-aliased resampling shape: "
            f"expected {(YOUR_CHANNELS, META_INPUT_SAMPLES)}, got {y.shape}"
        )

    return y.astype(np.float32, copy=False)


def preprocess_utah_trial(x_ct):
    """
    Utah:
        30 kHz -> anti-aliased 2 kHz -> 40 Hz high-pass -> optional Utah scale.
    """
    y = resample_utah_30k_to_2k(x_ct)
    y = highpass_numpy_channels_time(y)
    y = y * float(UTAH_INPUT_SCALE)
    return y.astype(np.float32, copy=False)


# ------------------------------------------------------------
# Load and freeze the official Meta model
# ------------------------------------------------------------

if not CKPT_PATH.exists():
    raise FileNotFoundError(f"Could not find Meta checkpoint:\n{CKPT_PATH}")

meta_model = DiscreteGesturesArchitecture(output_channels=NUM_META_CLASSES)

ckpt = torch.load(CKPT_PATH, map_location="cpu", weights_only=False)

if "state_dict" not in ckpt:
    raise KeyError("Checkpoint is missing 'state_dict'.")

state = ckpt["state_dict"]

network_state = {
    key.replace("network.", "", 1): value
    for key, value in state.items()
    if key.startswith("network.")
}

if not network_state:
    network_state = state

meta_model.load_state_dict(network_state, strict=True)

for parameter in meta_model.parameters():
    parameter.requires_grad = False

meta_model.eval()

# Verify the imported official architecture.
if not hasattr(meta_model, "compression"):
    raise RuntimeError("Meta model has no compression module.")

compression_range = float(getattr(meta_model.compression, "range", np.nan))
compression_midpoint = float(getattr(meta_model.compression, "midpoint", np.nan))

if not np.isclose(compression_range, 64.0):
    raise RuntimeError(
        f"Expected Meta compression range=64, got {compression_range}"
    )

if not np.isclose(compression_midpoint, 32.0):
    raise RuntimeError(
        f"Expected Meta compression midpoint=32, got {compression_midpoint}"
    )

META_LEFT_CONTEXT = int(meta_model.left_context)
META_OUTPUT_STRIDE = int(meta_model.stride)
EXPECTED_META_OUTPUT_SAMPLES = len(
    range(META_LEFT_CONTEXT, META_INPUT_SAMPLES, META_OUTPUT_STRIDE)
)


# ------------------------------------------------------------
# Regularized 32-channel -> 16-channel TCN adapter
# ------------------------------------------------------------

def make_group_norm(num_channels, requested_groups=8):
    groups = min(requested_groups, num_channels)

    while groups > 1 and num_channels % groups != 0:
        groups -= 1

    return nn.GroupNorm(groups, num_channels)


def make_wn_conv1d(
    in_channels,
    out_channels,
    kernel_size,
    padding=0,
    dilation=1,
    groups=1,
    bias=True,
):
    layer = nn.Conv1d(
        in_channels,
        out_channels,
        kernel_size=kernel_size,
        padding=padding,
        dilation=dilation,
        groups=groups,
        bias=bias,
    )
    return weight_norm(layer)


class DepthwiseSeparableTemporalConv(nn.Module):
    def __init__(
        self,
        channels,
        kernel_size=5,
        dilation=1,
        norm_groups=8,
    ):
        super().__init__()

        padding = dilation * (kernel_size - 1) // 2

        self.depthwise = make_wn_conv1d(
            channels,
            channels,
            kernel_size=kernel_size,
            padding=padding,
            dilation=dilation,
            groups=channels,
            bias=False,
        )

        self.pointwise = make_wn_conv1d(
            channels,
            channels,
            kernel_size=1,
            bias=False,
        )

        self.norm = make_group_norm(channels, norm_groups)
        self.activation = nn.SiLU()

    def forward(self, x):
        x = self.depthwise(x)
        x = self.pointwise(x)
        x = self.norm(x)
        return self.activation(x)


class RegularizedTCNBlock(nn.Module):
    def __init__(
        self,
        channels,
        kernel_size=5,
        dilation=1,
        norm_groups=8,
        initial_residual_scale=0.10,
    ):
        super().__init__()

        self.temporal_1 = DepthwiseSeparableTemporalConv(
            channels,
            kernel_size,
            dilation,
            norm_groups,
        )
        self.temporal_2 = DepthwiseSeparableTemporalConv(
            channels,
            kernel_size,
            dilation,
            norm_groups,
        )

        self.residual_scale = nn.Parameter(
            torch.tensor(float(initial_residual_scale))
        )

    def forward(self, x):
        residual = x
        out = self.temporal_1(x)
        out = self.temporal_2(out)
        return residual + self.residual_scale * out


class AdapterToMetaInput(nn.Module):
    """
    [B,32,2000] -> [B,16,2000].

    The adapter changes the channel representation without changing physical
    duration or sampling rate.
    """
    def __init__(
        self,
        input_channels=32,
        output_channels=16,
        hidden_channels=48,
    ):
        super().__init__()

        self.input_projection = nn.Sequential(
            make_wn_conv1d(
                input_channels,
                hidden_channels,
                kernel_size=1,
                bias=False,
            ),
            make_group_norm(hidden_channels, ADAPTER_GROUPS),
            nn.SiLU(),
        )

        self.temporal_stack = nn.Sequential(
            RegularizedTCNBlock(
                hidden_channels,
                ADAPTER_KERNEL_SIZE,
                dilation=1,
                norm_groups=ADAPTER_GROUPS,
                initial_residual_scale=INITIAL_RESIDUAL_SCALE,
            ),
            RegularizedTCNBlock(
                hidden_channels,
                ADAPTER_KERNEL_SIZE,
                dilation=2,
                norm_groups=ADAPTER_GROUPS,
                initial_residual_scale=INITIAL_RESIDUAL_SCALE,
            ),
            RegularizedTCNBlock(
                hidden_channels,
                ADAPTER_KERNEL_SIZE,
                dilation=4,
                norm_groups=ADAPTER_GROUPS,
                initial_residual_scale=INITIAL_RESIDUAL_SCALE,
            ),
            RegularizedTCNBlock(
                hidden_channels,
                ADAPTER_KERNEL_SIZE,
                dilation=8,
                norm_groups=ADAPTER_GROUPS,
                initial_residual_scale=INITIAL_RESIDUAL_SCALE,
            ),
        )

        self.output_projection = make_wn_conv1d(
            hidden_channels,
            output_channels,
            kernel_size=1,
            bias=True,
        )

        self.output_gain = nn.Parameter(
            torch.tensor(float(INITIAL_OUTPUT_GAIN))
        )

    def forward(self, x):
        if x.ndim != 3:
            raise ValueError(f"Expected [B,32,T], got {tuple(x.shape)}")

        if x.shape[1] != YOUR_CHANNELS:
            raise ValueError(
                f"Expected {YOUR_CHANNELS} Utah channels, got {tuple(x.shape)}"
            )

        if x.shape[-1] != META_INPUT_SAMPLES:
            raise ValueError(
                f"Expected one second at 2 kHz (T={META_INPUT_SAMPLES}), "
                f"got T={x.shape[-1]}"
            )

        x = self.input_projection(x)
        x = self.temporal_stack(x)
        x = self.output_projection(x)
        return self.output_gain * x


class FrozenMetaWithAdapter(nn.Module):
    def __init__(self, frozen_meta_model):
        super().__init__()

        self.adapter = AdapterToMetaInput(
            input_channels=YOUR_CHANNELS,
            output_channels=META_CHANNELS,
            hidden_channels=ADAPTER_HIDDEN_CHANNELS,
        )

        self.meta_model = frozen_meta_model

        for parameter in self.meta_model.parameters():
            parameter.requires_grad = False

        self.meta_model.eval()

    def train(self, mode=True):
        super().train(mode)
        self.meta_model.eval()
        return self

    def forward_with_adapter(self, x):
        adapter_output = self.adapter(x)
        raw_meta_output = self.meta_model(adapter_output)
        return raw_meta_output, adapter_output

    def forward(self, x):
        raw_meta_output, _ = self.forward_with_adapter(x)
        return raw_meta_output


def module_numeric_signature(module):
    """
    Lightweight frozen-model mutation check.
    """
    total_sum = 0.0
    total_sq_sum = 0.0
    total_count = 0

    with torch.no_grad():
        for tensor in module.state_dict().values():
            value = tensor.detach().double().cpu()
            total_sum += float(value.sum().item())
            total_sq_sum += float(value.square().sum().item())
            total_count += value.numel()

    return (total_count, total_sum, total_sq_sum)


model = FrozenMetaWithAdapter(meta_model).to(DEVICE)
initial_adapter_state = copy.deepcopy(model.adapter.state_dict())
initial_meta_signature = module_numeric_signature(model.meta_model)

total_params = sum(parameter.numel() for parameter in model.parameters())
trainable_params = sum(
    parameter.numel()
    for parameter in model.parameters()
    if parameter.requires_grad
)

if any(parameter.requires_grad for parameter in model.meta_model.parameters()):
    raise RuntimeError("The Meta backbone is not fully frozen.")

print("Using device:", DEVICE)
print("Physical timing:")
print(
    f"  Utah raw: {RAW_INPUT_SAMPLES} samples @ "
    f"{YOUR_SAMPLING_RATE} Hz = 1.000 s"
)
print(
    f"  Meta input: {META_INPUT_SAMPLES} samples @ "
    f"{META_SAMPLING_RATE} Hz = 1.000 s"
)
print(
    f"  Meta output: {EXPECTED_META_OUTPUT_SAMPLES} bins using "
    f"left_context={META_LEFT_CONTEXT}, stride={META_OUTPUT_STRIDE}"
)
print("Output mapping:", UTAH_TO_META_OUTPUTS)
print(
    "Meta internal compression:",
    f"{compression_range:g}*x/({compression_midpoint:g}+|x|)",
)
print("Total parameters:", f"{total_params:,}")
print("Trainable adapter parameters:", f"{trainable_params:,}")
print("Frozen parameters:", f"{total_params - trainable_params:,}")


Using device: cpu
Physical timing:
  Utah raw: 30000 samples @ 30000 Hz = 1.000 s
  Meta input: 2000 samples @ 2000 Hz = 1.000 s
  Meta output: 198 bins using left_context=20, stride=10
Output mapping: [5, 6, 7, 8, 4]
Meta internal compression: 64*x/(32+|x|)
Total parameters: 6,507,326
Trainable adapter parameters: 24,373
Frozen parameters: 6,482,953


In [2]:
# ============================================================
# CELL 2 — LOAD, CENTER-CROP, AND VALIDATE FIXED UTAH SPLITS
#
# IMPORTANT:
# The saved Utah entries are variable-length gesture-to-next-gesture
# segments. They are NOT already one-second trials.
#
# This cell:
#   1. finds the final active-valid label interval in trial-local coordinates,
#   2. extracts one 30,000-sample window centered on that interval,
#   3. applies the identical crop/pad to EMG, labels, and valid mask,
#   4. downsamples the one-second window exactly from 30 kHz to 2 kHz,
#   5. drops genuinely unusable TRAIN trials only,
#   6. hard-stops if any validation/test trial is unusable.
# ============================================================

from collections import defaultdict

import torch
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader


if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"Could not find labeled dataset:\n{DATA_PATH}"
    )


# ------------------------------------------------------------
# Orientation helpers
# ------------------------------------------------------------

def force_emg_time_channels(x):
    """Return Utah EMG as [time, 32]."""
    x = torch.as_tensor(x).float()

    if x.ndim != 2:
        raise ValueError(f"Expected 2D EMG, got {tuple(x.shape)}")

    if x.shape[1] == YOUR_CHANNELS:
        return x.contiguous()

    if x.shape[0] == YOUR_CHANNELS:
        return x.transpose(0, 1).contiguous()

    raise ValueError(
        f"Could not identify {YOUR_CHANNELS}-channel EMG orientation: "
        f"{tuple(x.shape)}"
    )


def force_labels_time_classes(x, expected_time):
    """Return labels as [time, classes]."""
    x = torch.as_tensor(x).float()

    if x.ndim != 2:
        raise ValueError(f"Expected 2D labels, got {tuple(x.shape)}")

    candidates = []

    if x.shape[0] == expected_time and x.shape[1] >= AO_KEPT_CLASSES:
        candidates.append(x.contiguous())

    if x.shape[1] == expected_time and x.shape[0] >= AO_KEPT_CLASSES:
        candidates.append(x.transpose(0, 1).contiguous())

    if len(candidates) == 1:
        return candidates[0]

    if len(candidates) > 1:
        raise ValueError(
            f"Ambiguous label orientation for shape {tuple(x.shape)} "
            f"and expected_time={expected_time}."
        )

    # Fallback when EMG and labels differ slightly in length.
    if x.shape[0] > x.shape[1] and x.shape[1] >= AO_KEPT_CLASSES:
        return x.contiguous()

    if x.shape[1] > x.shape[0] and x.shape[0] >= AO_KEPT_CLASSES:
        return x.transpose(0, 1).contiguous()

    raise ValueError(
        f"Could not identify label orientation: {tuple(x.shape)}"
    )


# ------------------------------------------------------------
# Trial-local active interval and one-second extraction
# ------------------------------------------------------------

def get_common_trial_arrays(trial):
    """
    Return aligned trial-local arrays:
        emg_tc:    [T,32]
        labels_tc: [T,C]
        valid_t:   [T]
    """
    if "ns5_vector" not in trial:
        raise KeyError("Trial is missing 'ns5_vector'.")

    if "trainKin" not in trial:
        raise KeyError("Trial is missing 'trainKin'.")

    emg_tc = force_emg_time_channels(trial["ns5_vector"])
    labels_tc = force_labels_time_classes(
        trial["trainKin"],
        expected_time=emg_tc.shape[0],
    )

    if "valid_mask" in trial:
        valid_t = torch.as_tensor(
            trial["valid_mask"]
        ).float().reshape(-1)
    else:
        valid_t = torch.ones(
            emg_tc.shape[0],
            dtype=torch.float32,
        )

    common_length = min(
        emg_tc.shape[0],
        labels_tc.shape[0],
        valid_t.shape[0],
    )

    if common_length <= 0:
        raise ValueError("Trial has no common EMG/label/mask samples.")

    return (
        emg_tc[:common_length].contiguous(),
        labels_tc[:common_length].contiguous(),
        valid_t[:common_length].contiguous(),
    )


def find_active_valid_interval(labels_tc, valid_t):
    """
    Find the first and last trial-local samples where:
        any Utah gesture label is active
        AND
        valid_mask == 1

    Returns None when the final saved trial has no usable active label.
    """
    active_t = (
        labels_tc[:, :AO_KEPT_CLASSES].amax(dim=1) > 0.5
    )
    valid_bool = valid_t > 0.5
    active_valid_t = active_t & valid_bool

    indices = torch.nonzero(
        active_valid_t,
        as_tuple=False,
    ).reshape(-1)

    if indices.numel() == 0:
        return None

    start = int(indices[0].item())
    end_exclusive = int(indices[-1].item()) + 1

    return start, end_exclusive


def extract_centered_time_window(
    x,
    center_index,
    target_samples,
    pad_value=0.0,
):
    """
    Extract one time-first window centered on `center_index`.

    If the full trial is at least target_samples long, the window is shifted
    at the boundaries so it contains only real samples.

    If the full trial is shorter than target_samples, it is padded while
    keeping the active interval center aligned to the middle of the output.

    Returns:
        output
        source_start
        source_end
        left_pad
        right_pad
    """
    if x.ndim < 1:
        raise ValueError("Input must have a time dimension.")

    total_samples = int(x.shape[0])
    target_samples = int(target_samples)
    center_index = int(center_index)

    if total_samples <= 0:
        raise ValueError("Cannot crop an empty trial.")

    if not 0 <= center_index < total_samples:
        raise ValueError(
            f"center_index={center_index} is outside trial length "
            f"{total_samples}."
        )

    if total_samples >= target_samples:
        source_start = center_index - target_samples // 2
        source_start = max(0, source_start)
        source_start = min(source_start, total_samples - target_samples)
        source_end = source_start + target_samples

        return (
            x[source_start:source_end].contiguous(),
            source_start,
            source_end,
            0,
            0,
        )

    # Short trial: preserve active-center alignment with explicit padding.
    desired_start = center_index - target_samples // 2
    desired_end = desired_start + target_samples

    source_start = max(0, desired_start)
    source_end = min(total_samples, desired_end)

    left_pad = source_start - desired_start
    copied = source_end - source_start
    right_pad = target_samples - left_pad - copied

    output_shape = (target_samples,) + tuple(x.shape[1:])
    output = torch.full(
        output_shape,
        float(pad_value),
        dtype=x.dtype,
    )

    destination_start = left_pad
    destination_end = destination_start + copied
    output[destination_start:destination_end] = x[source_start:source_end]

    return (
        output.contiguous(),
        source_start,
        source_end,
        left_pad,
        right_pad,
    )


def extract_one_second_active_centered_trial(trial):
    """
    Apply one identical trial-local crop/pad to EMG, labels, and valid mask.
    """
    emg_tc, labels_tc, valid_t = get_common_trial_arrays(trial)

    interval = find_active_valid_interval(labels_tc, valid_t)

    if interval is None:
        return None

    active_start, active_end = interval
    active_span = active_end - active_start

    if active_span > RAW_INPUT_SAMPLES:
        gesture = int(trial.get("gesture", -1))
        trial_num = int(trial.get("trial_num", -1))
        raise ValueError(
            f"G{gesture} trial {trial_num} has an active-valid interval "
            f"of {active_span} samples, longer than the one-second "
            f"window ({RAW_INPUT_SAMPLES})."
        )

    active_center = (active_start + active_end - 1) // 2

    emg_window, source_start, source_end, left_pad, right_pad = (
        extract_centered_time_window(
            emg_tc,
            center_index=active_center,
            target_samples=RAW_INPUT_SAMPLES,
            pad_value=0.0,
        )
    )

    labels_window, labels_source_start, labels_source_end, labels_left_pad, labels_right_pad = (
        extract_centered_time_window(
            labels_tc,
            center_index=active_center,
            target_samples=RAW_INPUT_SAMPLES,
            pad_value=0.0,
        )
    )

    valid_window, valid_source_start, valid_source_end, valid_left_pad, valid_right_pad = (
        extract_centered_time_window(
            valid_t,
            center_index=active_center,
            target_samples=RAW_INPUT_SAMPLES,
            pad_value=0.0,
        )
    )

    crop_metadata = {
        "original_length": int(emg_tc.shape[0]),
        "active_start": active_start,
        "active_end": active_end,
        "active_span": active_span,
        "active_center": active_center,
        "source_start": source_start,
        "source_end": source_end,
        "left_pad": left_pad,
        "right_pad": right_pad,
    }

    consistency_values = {
        (
            source_start,
            source_end,
            left_pad,
            right_pad,
        ),
        (
            labels_source_start,
            labels_source_end,
            labels_left_pad,
            labels_right_pad,
        ),
        (
            valid_source_start,
            valid_source_end,
            valid_left_pad,
            valid_right_pad,
        ),
    }

    if len(consistency_values) != 1:
        raise RuntimeError(
            "EMG, labels, and valid mask did not receive the same crop."
        )

    if emg_window.shape != (RAW_INPUT_SAMPLES, YOUR_CHANNELS):
        raise RuntimeError(
            f"Unexpected cropped EMG shape: {tuple(emg_window.shape)}"
        )

    if labels_window.shape[0] != RAW_INPUT_SAMPLES:
        raise RuntimeError(
            f"Unexpected cropped label shape: {tuple(labels_window.shape)}"
        )

    if valid_window.shape != (RAW_INPUT_SAMPLES,):
        raise RuntimeError(
            f"Unexpected cropped valid-mask shape: "
            f"{tuple(valid_window.shape)}"
        )

    return emg_window, labels_window, valid_window, crop_metadata


# ------------------------------------------------------------
# Exact 30 kHz -> 2 kHz label/mask reduction
# ------------------------------------------------------------

def downsample_binary_labels_30k_to_2k(labels_tc):
    """
    Exact 15:1 block max pooling.

    A positive label survives when any corresponding 30 kHz sample is positive.
    """
    if labels_tc.shape[0] != RAW_INPUT_SAMPLES:
        raise ValueError(
            f"Expected {RAW_INPUT_SAMPLES} label samples, got "
            f"{tuple(labels_tc.shape)}"
        )

    labels_ct = labels_tc.transpose(0, 1).unsqueeze(0)

    pooled = F.max_pool1d(
        labels_ct,
        kernel_size=UTAH_TO_META_DOWNSAMPLE,
        stride=UTAH_TO_META_DOWNSAMPLE,
    )

    pooled_tc = pooled.squeeze(0).transpose(0, 1).contiguous()

    if pooled_tc.shape[0] != META_INPUT_SAMPLES:
        raise RuntimeError(
            f"Label downsampling produced {pooled_tc.shape[0]} samples."
        )

    return pooled_tc


def downsample_valid_mask_30k_to_2k(valid_t):
    """
    Exact conservative 15:1 validity reduction.

    A 2 kHz sample is valid only when every contributing 30 kHz sample is valid.
    """
    if valid_t.shape[0] != RAW_INPUT_SAMPLES:
        raise ValueError(
            f"Expected {RAW_INPUT_SAMPLES} valid-mask samples, got "
            f"{tuple(valid_t.shape)}"
        )

    invalid = (1.0 - valid_t.clamp(0.0, 1.0)).reshape(1, 1, -1)

    pooled_invalid = F.max_pool1d(
        invalid,
        kernel_size=UTAH_TO_META_DOWNSAMPLE,
        stride=UTAH_TO_META_DOWNSAMPLE,
    )

    valid_2k = 1.0 - pooled_invalid.reshape(-1)

    if valid_2k.shape[0] != META_INPUT_SAMPLES:
        raise RuntimeError(
            f"Valid-mask downsampling produced {valid_2k.shape[0]} samples."
        )

    return valid_2k.float().contiguous()


# ------------------------------------------------------------
# Load and structurally validate saved splits
# ------------------------------------------------------------

saved = torch.load(
    DATA_PATH,
    map_location="cpu",
    weights_only=False,
)

for split_name in ["train", "val", "test"]:
    if split_name not in saved:
        raise KeyError(
            f"Dataset is missing fixed split '{split_name}'. "
            f"Available keys: {list(saved.keys())}"
        )


def split_gesture_counts(trials):
    counts = {
        class_index: 0
        for class_index in range(AO_KEPT_CLASSES)
    }

    for trial in trials:
        gesture = int(trial.get("gesture", -1))
        if gesture in counts:
            counts[gesture] += 1

    return counts


def split_trial_keys(trials):
    keys = set()

    for index, trial in enumerate(trials):
        gesture = int(trial.get("gesture", -1))
        trial_num = int(trial.get("trial_num", index))
        key = (gesture, trial_num)

        if key in keys:
            raise ValueError(
                f"Duplicate trial key within split: {key}"
            )

        keys.add(key)

    return keys


for split_name, expected_size in EXPECTED_SPLIT_SIZES.items():
    actual_size = len(saved[split_name])

    if actual_size != expected_size:
        raise ValueError(
            f"Expected {expected_size} {split_name} trials, "
            f"got {actual_size}."
        )

    counts = split_gesture_counts(saved[split_name])
    missing_classes = [
        class_index
        for class_index, count in counts.items()
        if count == 0
    ]

    if missing_classes:
        raise ValueError(
            f"{split_name} split is missing gesture classes: "
            f"{missing_classes}"
        )


train_keys = split_trial_keys(saved["train"])
val_keys = split_trial_keys(saved["val"])
test_keys = split_trial_keys(saved["test"])

if train_keys & val_keys:
    raise ValueError("Train and validation splits overlap.")

if train_keys & test_keys:
    raise ValueError("Train and test splits overlap.")

if val_keys & test_keys:
    raise ValueError("Validation and test splits overlap.")


# ------------------------------------------------------------
# Pre-loader audit
# ------------------------------------------------------------

true_no_active_trials = []
post_crop_failures = []
crop_summaries = defaultdict(list)
usable_trials = defaultdict(list)

for split_name in ["train", "val", "test"]:
    for index, trial in enumerate(saved[split_name]):
        gesture = int(trial.get("gesture", -1))
        trial_num = int(trial.get("trial_num", index))

        extracted = extract_one_second_active_centered_trial(trial)

        if extracted is None:
            emg_tc, labels_tc, valid_t = get_common_trial_arrays(trial)

            raw_active = (
                labels_tc[:, :AO_KEPT_CLASSES].amax(dim=1) > 0.5
            )
            raw_valid = valid_t > 0.5

            true_no_active_trials.append(
                {
                    "split": split_name,
                    "gesture": gesture,
                    "trial_num": trial_num,
                    "length": int(emg_tc.shape[0]),
                    "raw_active_samples": int(raw_active.sum().item()),
                    "raw_valid_samples": int(raw_valid.sum().item()),
                    "active_valid_overlap": int(
                        (raw_active & raw_valid).sum().item()
                    ),
                }
            )
            continue

        _, labels_window, valid_window, crop_metadata = extracted

        labels_2k_tc = downsample_binary_labels_30k_to_2k(
            labels_window[:, :AO_KEPT_CLASSES]
        )
        valid_2k = downsample_valid_mask_30k_to_2k(
            valid_window
        )

        active_valid_2k = (
            labels_2k_tc.amax(dim=1) > 0.5
        ) & (valid_2k > 0.5)

        crop_summaries[split_name].append(crop_metadata)

        if int(active_valid_2k.sum().item()) == 0:
            post_crop_failures.append(
                {
                    "split": split_name,
                    "gesture": gesture,
                    "trial_num": trial_num,
                    **crop_metadata,
                }
            )
            continue

        usable_trials[split_name].append(trial)


print("=" * 88)
print("ONE-SECOND ACTIVE-CENTERED CROP AUDIT")
print("=" * 88)

for split_name in ["train", "val", "test"]:
    summaries = crop_summaries[split_name]

    if summaries:
        active_spans = [
            item["active_span"]
            for item in summaries
        ]
        left_pads = [
            item["left_pad"]
            for item in summaries
        ]
        right_pads = [
            item["right_pad"]
            for item in summaries
        ]

        print(
            f"{split_name:>5}: "
            f"{len(summaries)} usable before final validation | "
            f"active span min/median/max = "
            f"{min(active_spans)}/"
            f"{int(torch.tensor(active_spans).float().median().item())}/"
            f"{max(active_spans)} | "
            f"padded trials = "
            f"{sum((l > 0 or r > 0) for l, r in zip(left_pads, right_pads))}"
        )
    else:
        print(f"{split_name:>5}: 0 usable before final validation")


if true_no_active_trials:
    print("\nGENUINELY UNUSABLE BEFORE CROPPING:")
    for item in true_no_active_trials:
        print(
            f"  {item['split']:>5} | "
            f"G{item['gesture']} trial {item['trial_num']} | "
            f"length={item['length']} | "
            f"active={item['raw_active_samples']} | "
            f"valid={item['raw_valid_samples']} | "
            f"active-valid={item['active_valid_overlap']}"
        )
else:
    print("\nNo genuinely unusable trials found before cropping.")


if post_crop_failures:
    print("\nFAILURES INTRODUCED BY CROP/DOWNSAMPLING:")
    for item in post_crop_failures:
        print(
            f"  {item['split']:>5} | "
            f"G{item['gesture']} trial {item['trial_num']} | "
            f"active span={item['active_span']} | "
            f"source=[{item['source_start']}:{item['source_end']}] | "
            f"pad=({item['left_pad']},{item['right_pad']})"
        )
else:
    print("\nNo usable trial lost its active label after centered cropping/downsampling.")


# Training trials without a meaningful active-valid burst are intentionally
# excluded. Validation and test trials are protected: losing even one is an
# evaluation failure and stops the notebook.
train_unusable = [
    item for item in true_no_active_trials
    if item["split"] == "train"
]
holdout_unusable = [
    item for item in true_no_active_trials
    if item["split"] in {"val", "test"}
]
holdout_post_crop_failures = [
    item for item in post_crop_failures
    if item["split"] in {"val", "test"}
]
train_post_crop_failures = [
    item for item in post_crop_failures
    if item["split"] == "train"
]

if holdout_unusable or holdout_post_crop_failures:
    raise RuntimeError(
        "\nA validation or test trial is unusable. Holdout trials must never be "
        "silently excluded; repair or replace the affected fixed holdout "
        "trial in the dataset-creation notebook."
    )

if train_post_crop_failures:
    raise RuntimeError(
        "\nAt least one otherwise usable training trial lost its label during "
        "cropping/downsampling. This indicates a loader bug and must not be "
        "silently excluded."
    )

filtered_train_trials = list(usable_trials["train"])
filtered_val_trials = list(usable_trials["val"])
filtered_test_trials = list(usable_trials["test"])

if len(filtered_val_trials) != EXPECTED_SPLIT_SIZES["val"]:
    raise RuntimeError(
        f"Validation must retain all {EXPECTED_SPLIT_SIZES['val']} fixed "
        f"trials, but {len(filtered_val_trials)} remain."
    )

if len(filtered_test_trials) != EXPECTED_SPLIT_SIZES["test"]:
    raise RuntimeError(
        f"Test must retain all {EXPECTED_SPLIT_SIZES['test']} fixed trials, "
        f"but {len(filtered_test_trials)} remain."
    )

filtered_train_counts = split_gesture_counts(filtered_train_trials)
missing_train_classes = [
    class_index
    for class_index, count in filtered_train_counts.items()
    if count == 0
]

if missing_train_classes:
    raise RuntimeError(
        "Dropping unusable training trials removed all examples for "
        f"classes {missing_train_classes}."
    )

print(
    f"\nExcluded {len(train_unusable)} unusable training trials; "
    "validation and test remain complete."
)
print("Filtered training counts:", filtered_train_counts)


# ------------------------------------------------------------
# Dataset and loaders
# ------------------------------------------------------------

class GestureOneSecond2kHzDataset(Dataset):
    """
    One active-centered one-second trial per item.

    Returns:
        emg:        [32,2000]
        target:     [5,2000]
        valid_mask: [2000]
        gesture:    scalar 0..4
        trial_num:  scalar
    """
    def __init__(self, trials, split_name):
        self.trials = list(trials)
        self.split_name = str(split_name)

    def __len__(self):
        return len(self.trials)

    def __getitem__(self, index):
        trial = self.trials[index]
        gesture = int(trial.get("gesture", -1))
        trial_num = int(trial.get("trial_num", index))

        if not 0 <= gesture < AO_KEPT_CLASSES:
            raise ValueError(
                f"{self.split_name} trial {trial_num} has invalid "
                f"gesture={gesture}; expected 0..{AO_KEPT_CLASSES - 1}."
            )

        extracted = extract_one_second_active_centered_trial(trial)

        if extracted is None:
            raise ValueError(
                f"{self.split_name} G{gesture} trial {trial_num} has no "
                "active-valid samples in its complete saved segment."
            )

        emg_tc, labels_tc, valid_t, _ = extracted

        emg_ct_np = emg_tc.transpose(0, 1).cpu().numpy()
        emg_2k_ct_np = preprocess_utah_trial(emg_ct_np)
        emg_2k_ct = torch.from_numpy(emg_2k_ct_np).float()

        target_2k_tc = downsample_binary_labels_30k_to_2k(
            labels_tc[:, :AO_KEPT_CLASSES]
        )
        target_2k_ct = target_2k_tc.transpose(0, 1).contiguous()

        valid_2k = downsample_valid_mask_30k_to_2k(valid_t)

        if not torch.isfinite(emg_2k_ct).all():
            raise ValueError(
                f"{self.split_name} G{gesture} trial {trial_num} "
                "contains non-finite EMG."
            )

        if not torch.isfinite(target_2k_ct).all():
            raise ValueError(
                f"{self.split_name} G{gesture} trial {trial_num} "
                "contains non-finite labels."
            )

        return (
            emg_2k_ct,
            target_2k_ct,
            valid_2k,
            torch.tensor(gesture, dtype=torch.long),
            torch.tensor(trial_num, dtype=torch.long),
        )


train_dataset = GestureOneSecond2kHzDataset(
    filtered_train_trials,
    "train",
)
val_dataset = GestureOneSecond2kHzDataset(
    filtered_val_trials,
    "val",
)
test_dataset = GestureOneSecond2kHzDataset(
    filtered_test_trials,
    "test",
)

train_generator = torch.Generator()
train_generator.manual_seed(SEED)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    drop_last=False,
    num_workers=0,
    generator=train_generator,
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    drop_last=False,
    num_workers=0,
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    drop_last=False,
    num_workers=0,
)


# ------------------------------------------------------------
# Final loader validation
# ------------------------------------------------------------

def validate_trial_label_consistency(dataset):
    errors = []

    for index in range(len(dataset)):
        _, target, valid_mask, gesture, trial_num = dataset[index]

        active = (
            target.amax(dim=0) > 0.5
        ) & (valid_mask > 0.5)

        if int(active.sum().item()) == 0:
            errors.append(
                f"{dataset.split_name} G{int(gesture)} trial "
                f"{int(trial_num)} has no active-valid 2 kHz samples."
            )
            continue

        active_classes = target[:, active].argmax(dim=0)
        dominant_class = int(
            torch.bincount(
                active_classes,
                minlength=AO_KEPT_CLASSES,
            ).argmax().item()
        )

        if dominant_class != int(gesture.item()):
            errors.append(
                f"{dataset.split_name} G{int(gesture)} trial "
                f"{int(trial_num)} has dominant label G{dominant_class}."
            )

    if errors:
        print("\nFINAL LABEL-CONSISTENCY FAILURES:")
        for error in errors:
            print(" ", error)

        raise RuntimeError(
            f"{len(errors)} trial(s) failed final label consistency."
        )


validate_trial_label_consistency(train_dataset)
validate_trial_label_consistency(val_dataset)
validate_trial_label_consistency(test_dataset)


print("\nFixed Utah loaders validated.")
print(
    "  Crop policy: one second centered on each final "
    "trial-local active-valid label interval"
)
print(
    "  Labels already shifted 100 ms:",
    UTAH_LABELS_ALREADY_SHIFTED_100_MS,
)
print(
    "  Train:",
    len(train_dataset),
    split_gesture_counts(filtered_train_trials),
)
print(
    "  Validation:",
    len(val_dataset),
    split_gesture_counts(filtered_val_trials),
)
print(
    "  Test:",
    len(test_dataset),
    split_gesture_counts(filtered_test_trials),
)

example_batch = next(iter(train_loader))

print("\nExample batch:")
print("  EMG:", tuple(example_batch[0].shape))
print("  Target:", tuple(example_batch[1].shape))
print("  Valid mask:", tuple(example_batch[2].shape))
print("  Gesture IDs:", example_batch[3].tolist())


ValueError: Expected 80 train trials, got 43.

In [ ]:
# ============================================================
# CELL 3 — EXACT META TARGET ALIGNMENT + MAPPED TASK BCE
# ============================================================

import torch
import torch.nn.functional as F


META_OUTPUT_INDEX_TENSOR = torch.tensor(
    UTAH_TO_META_OUTPUTS,
    dtype=torch.long,
    device=DEVICE,
)


def map_meta_outputs_to_utah(raw_meta_output):
    """
    Meta [B,9,T] -> Utah order [B,5,T]:
        left, right, up, down, thumb press/tap.
    """
    if raw_meta_output.ndim != 3:
        raise ValueError(
            f"Expected Meta output [B,9,T], got "
            f"{tuple(raw_meta_output.shape)}"
        )

    if raw_meta_output.shape[1] != NUM_META_CLASSES:
        raise ValueError(
            f"Expected {NUM_META_CLASSES} Meta outputs, got "
            f"{tuple(raw_meta_output.shape)}"
        )

    selected = raw_meta_output.index_select(
        dim=1,
        index=META_OUTPUT_INDEX_TENSOR,
    )

    if META_OUTPUT_IS_PROBABILITY:
        selected = torch.logit(
            selected.clamp(1e-5, 1.0 - 1e-5)
        )

    return selected


def align_target_and_mask_to_logits(target, valid_mask, output_length):
    """
    Match the official Meta training alignment exactly:

        target[..., left_context::stride]

    For the official discrete-gesture model this is:
        target[..., 20::10] -> 198 bins for a 2,000-sample input.
    """
    if target.ndim != 3:
        raise ValueError(
            f"Expected target [B,5,T], got {tuple(target.shape)}"
        )

    if valid_mask.ndim != 2:
        raise ValueError(
            f"Expected valid_mask [B,T], got {tuple(valid_mask.shape)}"
        )

    aligned_target = target[
        ...,
        META_LEFT_CONTEXT::META_OUTPUT_STRIDE,
    ]

    aligned_valid = valid_mask[
        ...,
        META_LEFT_CONTEXT::META_OUTPUT_STRIDE,
    ]

    if aligned_target.shape[-1] != output_length:
        raise RuntimeError(
            "Target alignment does not match Meta output length: "
            f"target={aligned_target.shape[-1]}, output={output_length}."
        )

    if aligned_valid.shape[-1] != output_length:
        raise RuntimeError(
            "Valid-mask alignment does not match Meta output length: "
            f"mask={aligned_valid.shape[-1]}, output={output_length}."
        )

    return aligned_target, aligned_valid


def ao_unpack_batch(batch):
    emg, target, valid_mask, gesture, trial_num = batch

    return (
        emg.to(DEVICE).float(),
        target.to(DEVICE).float(),
        valid_mask.to(DEVICE).float(),
        gesture.to(DEVICE).long(),
        trial_num.to(DEVICE).long(),
    )


def active_only_multilabel_bce(logits, target, valid_mask):
    """
    Independent five-output BCE over active valid Utah bins.

    Negative classes are still penalized within every active bin. Rest bins
    are intentionally excluded so that the small dataset is not dominated by
    all-zero time points.
    """
    active = target.max(dim=1).values > 0.5
    valid = valid_mask > 0.5
    keep = active & valid

    if int(keep.sum().item()) == 0:
        raise RuntimeError("Batch contains no active valid target bins.")

    per_class_loss = F.binary_cross_entropy_with_logits(
        logits,
        target,
        reduction="none",
    )

    keep_expanded = keep.unsqueeze(1).expand_as(per_class_loss)

    return per_class_loss[keep_expanded].mean()


print("Loss and alignment functions ready.")
print("  Exact target alignment:", f"{META_LEFT_CONTEXT}::{META_OUTPUT_STRIDE}")
print("  Mapped Meta outputs:", UTAH_TO_META_OUTPUTS)
print("  Task loss: active-only five-output BCEWithLogits")


In [ ]:
# ============================================================
# CELL 4 — PRETRAINING SIGNAL, ALIGNMENT, AND FREEZE DIAGNOSTIC
# Run this before Cell 5. It does not update any model parameters.
# ============================================================

import torch


def signal_stats(name, tensor):
    x = torch.as_tensor(tensor).detach().float().cpu()

    finite = bool(torch.isfinite(x).all().item())
    stats = {
        "name": name,
        "shape": tuple(x.shape),
        "finite": finite,
        "mean": float(x.mean().item()),
        "std": float(x.std(unbiased=False).item()),
        "rms": float(torch.sqrt(x.square().mean()).item()),
        "min": float(x.min().item()),
        "max": float(x.max().item()),
    }

    print(
        f"{name:<34} shape={str(stats['shape']):<18} "
        f"mean={stats['mean']:+.3e} std={stats['std']:.3e} "
        f"rms={stats['rms']:.3e} "
        f"range=[{stats['min']:+.3e},{stats['max']:+.3e}] "
        f"finite={finite}"
    )

    return stats


utah_batch = next(iter(train_loader))
emg, target, valid_mask, gestures, trial_nums = ao_unpack_batch(utah_batch)

model.eval()

with torch.no_grad():
    raw_meta_output, adapter_output = model.forward_with_adapter(emg)
    mapped_logits = map_meta_outputs_to_utah(raw_meta_output)

    aligned_target, aligned_valid = align_target_and_mask_to_logits(
        target,
        valid_mask,
        mapped_logits.shape[-1],
    )

    task_loss = active_only_multilabel_bce(
        mapped_logits,
        aligned_target,
        aligned_valid,
    )

    compressed_adapter = model.meta_model.compression(adapter_output)


print("=" * 96)
print("SIGNAL SCALE DIAGNOSTIC")
print("=" * 96)
signal_stats("Utah input after 2 kHz + HP", emg)
signal_stats("Untrained adapter output", adapter_output)
signal_stats("Adapter after Meta compression", compressed_adapter)


print()
print("=" * 96)
print("MODEL, LOSS, AND ALIGNMENT CHECK")
print("=" * 96)
print("Utah input:", tuple(emg.shape))
print("Adapter / Meta input:", tuple(adapter_output.shape))
print("Raw Meta output:", tuple(raw_meta_output.shape))
print("Mapped output:", tuple(mapped_logits.shape))
print("Aligned target:", tuple(aligned_target.shape))
print("Aligned valid mask:", tuple(aligned_valid.shape))
print("Mapping:", UTAH_TO_META_OUTPUTS)
print("Task BCE:", f"{float(task_loss):.6e}")
print(
    "Raw Meta output range:",
    f"[{float(raw_meta_output.min()):+.3e}, "
    f"{float(raw_meta_output.max()):+.3e}]",
)
print(
    "Frozen Meta parameters trainable:",
    sum(
        int(parameter.requires_grad)
        for parameter in model.meta_model.parameters()
    ),
)
print(
    "Trainable adapter parameter count:",
    sum(
        parameter.numel()
        for parameter in model.adapter.parameters()
        if parameter.requires_grad
    ),
)
print(
    "Meta internal compression:",
    f"{compression_range:g}*x/"
    f"({compression_midpoint:g}+|x|)",
)

preflight_errors = []

if raw_meta_output.shape[1] != NUM_META_CLASSES:
    preflight_errors.append("Meta output does not have nine classes.")

if mapped_logits.shape[1] != AO_KEPT_CLASSES:
    preflight_errors.append("Mapped output does not have five classes.")

if mapped_logits.shape[-1] != EXPECTED_META_OUTPUT_SAMPLES:
    preflight_errors.append(
        "Meta output temporal length does not match the official alignment."
    )

if aligned_target.shape != mapped_logits.shape:
    preflight_errors.append(
        "Aligned target shape does not match mapped output shape."
    )

if aligned_valid.shape != (
    mapped_logits.shape[0],
    mapped_logits.shape[-1],
):
    preflight_errors.append(
        "Aligned valid-mask shape does not match mapped output timing."
    )

if not torch.isfinite(task_loss):
    preflight_errors.append("Task loss is non-finite.")

if not torch.isfinite(adapter_output).all():
    preflight_errors.append("Adapter output contains non-finite values.")

if not torch.isfinite(raw_meta_output).all():
    preflight_errors.append("Meta output contains non-finite values.")

if any(
    parameter.requires_grad
    for parameter in model.meta_model.parameters()
):
    preflight_errors.append("Meta backbone is not fully frozen.")

if META_OUTPUT_IS_PROBABILITY:
    preflight_errors.append(
        "META_OUTPUT_IS_PROBABILITY should be False for the official model."
    )

PREFLIGHT_PASSED = len(preflight_errors) == 0

if preflight_errors:
    print()
    print("PREFLIGHT ERRORS:")
    for error in preflight_errors:
        print(" -", error)
else:
    print()
    print("PREFLIGHT STRUCTURAL CHECKS PASSED.")


In [ ]:
# ============================================================
# CELL 5 — TRAIN ADAPTER ONLY + SELECT BEST VALIDATION EPOCH
# Test is evaluated once, after restoring the best adapter state.
# ============================================================

import copy
import csv
from pathlib import Path

import numpy as np
import torch
import matplotlib.pyplot as plt


# Preserve editable text in SVG and TrueType text in PDF for Adobe.
plt.rcParams["svg.fonttype"] = "none"
plt.rcParams["pdf.fonttype"] = 42
plt.rcParams["ps.fonttype"] = 42
plt.rcParams["font.family"] = "sans-serif"
plt.rcParams["font.sans-serif"] = [
    "Arial",
    "Helvetica",
    "DejaVu Sans",
]


def save_figure_adobe(
    figure,
    output_dir,
    stem,
):
    """
    Save one figure as editable SVG, vector PDF, and 300-dpi PNG.
    """
    output_dir = Path(output_dir)

    output_dir.mkdir(
        parents=True,
        exist_ok=True,
    )

    saved_paths = []

    for extension, extra_kwargs in (
        ("svg", {}),
        ("pdf", {}),
        ("png", {"dpi": 300}),
    ):
        path = output_dir / f"{stem}.{extension}"

        figure.savefig(
            path,
            bbox_inches="tight",
            pad_inches=0.05,
            facecolor="white",
            **extra_kwargs,
        )

        saved_paths.append(path)

    print(f"Saved Adobe-compatible figure: {stem}")
    for path in saved_paths:
        print(" ", path.resolve())


if "PREFLIGHT_PASSED" not in globals():
    raise RuntimeError("Run Cell 4 before training.")

if not PREFLIGHT_PASSED:
    raise RuntimeError(
        "Preflight structural checks failed. Fix Cell 4 errors first."
    )

def reset_adapter():
    model.adapter.load_state_dict(
        copy.deepcopy(initial_adapter_state)
    )
    model.to(DEVICE)


def build_adamw_parameter_groups(module, weight_decay):
    decay = []
    no_decay = []

    for name, parameter in module.named_parameters():
        if not parameter.requires_grad:
            continue

        name_lower = name.lower()

        if (
            parameter.ndim <= 1
            or name_lower.endswith(".bias")
            or "norm" in name_lower
            or "residual_scale" in name_lower
            or "output_gain" in name_lower
        ):
            no_decay.append(parameter)
        else:
            decay.append(parameter)

    if not decay and not no_decay:
        raise RuntimeError("No trainable adapter parameters found.")

    groups = []

    if decay:
        groups.append({
            "params": decay,
            "weight_decay": float(weight_decay),
        })

    if no_decay:
        groups.append({
            "params": no_decay,
            "weight_decay": 0.0,
        })

    return groups


def build_optimizer(lr=AO_LR):
    parameter_groups = build_adamw_parameter_groups(
        model.adapter,
        AO_WEIGHT_DECAY,
    )

    return torch.optim.AdamW(
        parameter_groups,
        lr=float(lr),
    )


def scheduled_lr(epoch_number):
    if epoch_number <= AO_WARMUP_EPOCHS:
        fraction = epoch_number / max(1, AO_WARMUP_EPOCHS)
        return AO_LR * fraction

    if epoch_number >= AO_DECAY_EPOCH:
        return AO_LR * AO_DECAY_FACTOR

    return AO_LR


def set_optimizer_lr(optimizer, lr):
    for parameter_group in optimizer.param_groups:
        parameter_group["lr"] = float(lr)


def set_model_mode(train):
    model.train(train)
    model.meta_model.eval()


def trial_predictions(logits, target, valid_mask, gestures):
    """
    Return one mean-logit prediction per complete trial.

    The mapped Meta outputs are trained with independent BCE, but the Utah
    task requires one gesture decision per trial. Therefore, logits are
    averaged across active-valid output bins and the largest mean logit is
    selected.
    """
    active = target.max(dim=1).values > 0.5
    valid = valid_mask > 0.5
    keep = active & valid

    results = []

    for batch_index in range(logits.shape[0]):
        trial_keep = keep[batch_index]

        if int(trial_keep.sum().item()) == 0:
            raise RuntimeError(
                f"Batch item {batch_index} has no active valid output bins."
            )

        true_class = int(gestures[batch_index].item())

        if not 0 <= true_class < AO_KEPT_CLASSES:
            raise ValueError(
                f"Invalid trial gesture id {true_class}."
            )

        mean_logits = logits[
            batch_index,
            :,
            trial_keep,
        ].mean(dim=1)

        prediction = int(mean_logits.argmax().item())

        # Softmax is used only to display normalized mutually exclusive scores.
        # Training remains independent-output BCEWithLogits.
        normalized_scores = torch.softmax(
            mean_logits,
            dim=0,
        ).detach().cpu()

        results.append({
            "batch_index": batch_index,
            "true": true_class,
            "prediction": prediction,
            "normalized_scores": normalized_scores,
            "active_bins": int(trial_keep.sum().item()),
        })

    return results


def run_epoch(loader, train):
    set_model_mode(train)

    task_loss_sum = 0.0
    batch_count = 0

    correct_trials = 0
    trial_total = 0
    wrong_trials = []

    raw_output_min = float("inf")
    raw_output_max = float("-inf")

    for batch in loader:
        emg, target, valid_mask, gestures, trial_nums = ao_unpack_batch(batch)

        if train:
            ao_optimizer.zero_grad(set_to_none=True)

        with torch.set_grad_enabled(train):
            raw_meta_output = model(emg)

            raw_output_min = min(
                raw_output_min,
                float(raw_meta_output.detach().min().item()),
            )
            raw_output_max = max(
                raw_output_max,
                float(raw_meta_output.detach().max().item()),
            )

            logits = map_meta_outputs_to_utah(raw_meta_output)

            target, valid_mask = align_target_and_mask_to_logits(
                target,
                valid_mask,
                logits.shape[-1],
            )

            task_loss = active_only_multilabel_bce(
                logits,
                target,
                valid_mask,
            )

            if not torch.isfinite(task_loss):
                raise FloatingPointError(
                    "Non-finite task loss detected."
                )

            if train:
                task_loss.backward()

                gradient_norm = torch.nn.utils.clip_grad_norm_(
                    [
                        parameter
                        for parameter in model.adapter.parameters()
                        if parameter.requires_grad
                    ],
                    max_norm=AO_GRAD_CLIP_NORM,
                )

                if not torch.isfinite(gradient_norm):
                    raise FloatingPointError(
                        "Non-finite adapter gradient norm detected."
                    )

                ao_optimizer.step()

        batch_results = trial_predictions(
            logits.detach(),
            target.detach(),
            valid_mask.detach(),
            gestures.detach(),
        )

        for result in batch_results:
            is_correct = (
                result["prediction"] == result["true"]
            )

            correct_trials += int(is_correct)
            trial_total += 1

            if not is_correct:
                batch_index = result["batch_index"]

                wrong_trials.append({
                    "trial_num": int(
                        trial_nums[batch_index].item()
                    ),
                    "true": int(result["true"]),
                    "prediction": int(
                        result["prediction"]
                    ),
                })

        task_loss_sum += float(task_loss.detach().item())
        batch_count += 1

    denominator = max(1, batch_count)

    return {
        "task_loss": task_loss_sum / denominator,
        "accuracy": correct_trials / max(1, trial_total),
        "num_trials": trial_total,
        "wrong_trials": wrong_trials,
        "raw_output_min": raw_output_min,
        "raw_output_max": raw_output_max,
    }


def detailed_trial_evaluation(loader, split_name):
    set_model_mode(False)

    confusion = torch.zeros(
        AO_KEPT_CLASSES,
        AO_KEPT_CLASSES,
        dtype=torch.long,
    )

    rows = []

    with torch.no_grad():
        for batch in loader:
            emg, target, valid_mask, gestures, trial_nums = ao_unpack_batch(batch)

            raw_meta_output = model(emg)
            logits = map_meta_outputs_to_utah(raw_meta_output)

            target, valid_mask = align_target_and_mask_to_logits(
                target,
                valid_mask,
                logits.shape[-1],
            )

            batch_results = trial_predictions(
                logits,
                target,
                valid_mask,
                gestures,
            )

            for result in batch_results:
                batch_index = result["batch_index"]
                true_class = result["true"]
                predicted_class = result["prediction"]

                confusion[true_class, predicted_class] += 1

                rows.append({
                    "trial_num": int(
                        trial_nums[batch_index].item()
                    ),
                    "true": true_class,
                    "predicted": predicted_class,
                    "normalized_scores": (
                        result["normalized_scores"].numpy()
                    ),
                    "active_bins": result["active_bins"],
                })

    correct = int(confusion.diag().sum().item())
    total = int(confusion.sum().item())
    accuracy = correct / max(1, total)

    print()
    print("=" * 80)
    print(f"{split_name.upper()} COMPLETE-TRIAL RESULTS")
    print("=" * 80)
    print(f"Accuracy: {correct}/{total} = {accuracy:.4f}")

    for row in rows:
        score_text = ", ".join(
            f"{AO_CLASS_NAMES[index]}="
            f"{row['normalized_scores'][index]:.3f}"
            for index in range(AO_KEPT_CLASSES)
        )

        print(
            f"trial {row['trial_num']:>3} | "
            f"true={AO_CLASS_NAMES[row['true']]} | "
            f"pred={AO_CLASS_NAMES[row['predicted']]} | "
            f"active bins={row['active_bins']} | "
            f"{score_text}"
        )

    return confusion, rows, accuracy


def plot_confusion(
    confusion,
    title,
    output_dir,
    export_stem,
):
    matrix = confusion.numpy()

    figure, axis = plt.subplots(
        figsize=(7, 6)
    )

    image = axis.imshow(
        matrix,
        interpolation="nearest",
        cmap="Blues",
    )

    figure.colorbar(
        image,
        ax=axis,
        label="Complete trials",
    )

    axis.set_title(title)
    axis.set_xlabel("Predicted gesture")
    axis.set_ylabel("True gesture")

    ticks = np.arange(AO_KEPT_CLASSES)
    short_names = [
        "left",
        "right",
        "up",
        "down",
        "press",
    ]

    axis.set_xticks(
        ticks,
        short_names,
        rotation=30,
        ha="right",
    )

    axis.set_yticks(
        ticks,
        short_names,
    )

    threshold = (
        matrix.max() / 2.0
        if matrix.size
        else 0.0
    )

    for row in range(
        matrix.shape[0]
    ):
        for column in range(
            matrix.shape[1]
        ):
            axis.text(
                column,
                row,
                str(matrix[row, column]),
                ha="center",
                va="center",
                color=(
                    "white"
                    if matrix[row, column] > threshold
                    else "black"
                ),
            )

    figure.tight_layout()

    save_figure_adobe(
        figure,
        output_dir,
        export_stem,
    )

    plt.show()

    return figure


reset_adapter()
ao_optimizer = build_optimizer(AO_LR)

history = {
    "train_task_loss": [],
    "val_task_loss": [],
    "train_accuracy": [],
    "val_accuracy": [],
}

best_adapter_state = None
best_epoch = None
best_val_accuracy = float("-inf")
best_val_task_loss = float("inf")


RESULTS_EXPORT_DIR = Path.cwd() / "figures"

RESULTS_EXPORT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

print("Figure export directory:")
print(RESULTS_EXPORT_DIR.resolve())
print()

print("Starting corrected fixed-split training.")
print("  Train trials:", len(train_loader.dataset))
print("  Validation trials:", len(val_loader.dataset))
print("  Test trials held blind:", len(test_loader.dataset))
print("  Mapped Meta outputs:", UTAH_TO_META_OUTPUTS)
print("  Meta backbone frozen:", True)

for epoch in range(1, AO_EPOCHS + 1):
    current_lr = scheduled_lr(epoch)
    set_optimizer_lr(ao_optimizer, current_lr)

    train_metrics = run_epoch(train_loader, train=True)
    val_metrics = run_epoch(val_loader, train=False)

    history["train_task_loss"].append(train_metrics["task_loss"])
    history["val_task_loss"].append(val_metrics["task_loss"])
    history["train_accuracy"].append(train_metrics["accuracy"])
    history["val_accuracy"].append(val_metrics["accuracy"])

    epoch_line = (
        f"Epoch {epoch:03d}/{AO_EPOCHS} | "
        f"lr {current_lr:.2e} | "
        f"train BCE {train_metrics['task_loss']:.4f} | "
        f"val BCE {val_metrics['task_loss']:.4f} | "
        f"train acc {train_metrics['accuracy']:.3f} | "
        f"val acc {val_metrics['accuracy']:.3f}"
    )

    # With five validation trials, 0.8 means exactly one trial is wrong.
    # Report only that error so repeated 0.8 epochs reveal whether the
    # same validation gesture is systematically misclassified.
    if np.isclose(val_metrics["accuracy"], 0.8):
        wrong_val_trials = val_metrics["wrong_trials"]

        if len(wrong_val_trials) == 1:
            wrong = wrong_val_trials[0]

            epoch_line += (
                " | wrong val: "
                f"G{wrong['true']} trial {wrong['trial_num']} "
                f"({AO_CLASS_NAMES[wrong['true']]} -> "
                f"{AO_CLASS_NAMES[wrong['prediction']]})"
            )
        else:
            epoch_line += (
                " | wrong val count mismatch: "
                f"{len(wrong_val_trials)}"
            )

    print(epoch_line)

    val_accuracy = val_metrics["accuracy"]
    val_loss = val_metrics["task_loss"]

    is_better = (
        val_accuracy > best_val_accuracy
        or (
            np.isclose(val_accuracy, best_val_accuracy)
            and val_loss < best_val_task_loss
        )
    )

    if is_better:
        best_val_accuracy = val_accuracy
        best_val_task_loss = val_loss
        best_epoch = epoch
        best_adapter_state = copy.deepcopy(
            model.adapter.state_dict()
        )

if best_adapter_state is None:
    raise RuntimeError("No best adapter state was recorded.")

model.adapter.load_state_dict(best_adapter_state)
model.to(DEVICE)
model.meta_model.eval()

print()
print(
    f"Restored best adapter from epoch {best_epoch}: "
    f"val accuracy={best_val_accuracy:.4f}, "
    f"val task BCE={best_val_task_loss:.4f}"
)

# Verify that training did not mutate the frozen Meta backbone.
final_meta_signature = module_numeric_signature(model.meta_model)

if final_meta_signature != initial_meta_signature:
    raise RuntimeError(
        "Frozen Meta backbone changed during adapter training."
    )

test_metrics = run_epoch(test_loader, train=False)

print()
print("=" * 80)
print("FINAL BLIND TEST — BEST VALIDATION-SELECTED ADAPTER")
print("=" * 80)
print("Best epoch:", best_epoch)
print("Test task BCE:", round(test_metrics["task_loss"], 4))
print("Test accuracy:", round(test_metrics["accuracy"], 4))

val_confusion, val_rows, val_accuracy = detailed_trial_evaluation(
    val_loader,
    "Validation",
)
test_confusion, test_rows, test_accuracy = detailed_trial_evaluation(
    test_loader,
    "Test",
)

plot_confusion(
    val_confusion,
    "Validation complete-trial confusion matrix",
    output_dir=RESULTS_EXPORT_DIR,
    export_stem="validation_confusion_matrix",
)

plot_confusion(
    test_confusion,
    "Test complete-trial confusion matrix",
    output_dir=RESULTS_EXPORT_DIR,
    export_stem="test_confusion_matrix",
)

epochs = np.arange(
    1,
    AO_EPOCHS + 1,
)


# ------------------------------------------------------------
# Training and validation task loss
# ------------------------------------------------------------

loss_figure, loss_axis = plt.subplots(
    figsize=(9, 5)
)

loss_axis.plot(
    epochs,
    history["train_task_loss"],
    label="Training task BCE",
)

loss_axis.plot(
    epochs,
    history["val_task_loss"],
    label="Validation task BCE",
)

loss_axis.axvline(
    best_epoch,
    linestyle="--",
    label=f"Best epoch {best_epoch}",
)

loss_axis.set(
    xlabel="Epoch",
    ylabel="BCE",
    title="Training and validation classification loss",
)

loss_axis.grid(
    True,
    alpha=0.3,
)

loss_axis.legend(
    frameon=False,
)

loss_figure.tight_layout()

save_figure_adobe(
    loss_figure,
    RESULTS_EXPORT_DIR,
    "training_validation_loss",
)

plt.show()


# ------------------------------------------------------------
# Training and validation complete-trial accuracy
# ------------------------------------------------------------

accuracy_figure, accuracy_axis = plt.subplots(
    figsize=(9, 5)
)

accuracy_axis.plot(
    epochs,
    history["train_accuracy"],
    label="Training accuracy",
)

accuracy_axis.plot(
    epochs,
    history["val_accuracy"],
    label="Validation accuracy",
)

accuracy_axis.axvline(
    best_epoch,
    linestyle="--",
    label=f"Best epoch {best_epoch}",
)

accuracy_axis.set(
    xlabel="Epoch",
    ylabel="Complete-trial accuracy",
    ylim=(0, 1.05),
    title="Training and validation accuracy",
)

accuracy_axis.grid(
    True,
    alpha=0.3,
)

accuracy_axis.legend(
    frameon=False,
)

accuracy_figure.tight_layout()

save_figure_adobe(
    accuracy_figure,
    RESULTS_EXPORT_DIR,
    "training_validation_accuracy",
)

plt.show()


# ------------------------------------------------------------
# Poster-ready validation/test summary
# ------------------------------------------------------------

summary_labels = [
    "Validation",
    "Test",
]

summary_values = [
    100.0 * val_accuracy,
    100.0 * test_accuracy,
]

summary_figure, summary_axis = plt.subplots(
    figsize=(6.5, 5)
)

summary_bars = summary_axis.bar(
    summary_labels,
    summary_values,
)

summary_axis.axhline(
    20.0,
    linestyle="--",
    linewidth=1.5,
    label="Five-class chance level",
)

summary_axis.set(
    ylabel="Complete-trial accuracy (%)",
    ylim=(0, 105),
    title="Held-out gesture accuracy",
)

summary_axis.legend(
    frameon=False,
)

for bar, value in zip(
    summary_bars,
    summary_values,
):
    summary_axis.text(
        bar.get_x() + bar.get_width() / 2,
        value + 2,
        f"{value:.0f}%",
        ha="center",
        va="bottom",
    )

summary_figure.tight_layout()

save_figure_adobe(
    summary_figure,
    RESULTS_EXPORT_DIR,
    "validation_test_accuracy_summary",
)

plt.show()


# ------------------------------------------------------------
# Export numerical values alongside the figures
# ------------------------------------------------------------

history_csv_path = (
    RESULTS_EXPORT_DIR
    / "training_history.csv"
)

with history_csv_path.open(
    "w",
    newline="",
    encoding="utf-8",
) as handle:
    writer = csv.writer(handle)

    writer.writerow([
        "epoch",
        "train_task_bce",
        "validation_task_bce",
        "train_accuracy",
        "validation_accuracy",
    ])

    for epoch_index in range(
        AO_EPOCHS
    ):
        writer.writerow([
            epoch_index + 1,
            history["train_task_loss"][epoch_index],
            history["val_task_loss"][epoch_index],
            history["train_accuracy"][epoch_index],
            history["val_accuracy"][epoch_index],
        ])


summary_csv_path = (
    RESULTS_EXPORT_DIR
    / "run_summary.csv"
)

with summary_csv_path.open(
    "w",
    newline="",
    encoding="utf-8",
) as handle:
    writer = csv.writer(handle)

    writer.writerow([
        "best_epoch",
        "best_validation_accuracy",
        "best_validation_task_bce",
        "final_validation_accuracy",
        "test_accuracy",
        "test_task_bce",
    ])

    writer.writerow([
        best_epoch,
        best_val_accuracy,
        best_val_task_loss,
        val_accuracy,
        test_accuracy,
        test_metrics["task_loss"],
    ])


def save_confusion_csv(
    matrix,
    filename,
):
    matrix = matrix.numpy()

    path = (
        RESULTS_EXPORT_DIR
        / filename
    )

    short_names = [
        "left",
        "right",
        "up",
        "down",
        "press",
    ]

    with path.open(
        "w",
        newline="",
        encoding="utf-8",
    ) as handle:
        writer = csv.writer(handle)

        writer.writerow([
            "true\\predicted",
            *short_names,
        ])

        for label, row in zip(
            short_names,
            matrix,
        ):
            writer.writerow([
                label,
                *row.tolist(),
            ])


save_confusion_csv(
    val_confusion,
    "validation_confusion_matrix.csv",
)

save_confusion_csv(
    test_confusion,
    "test_confusion_matrix.csv",
)

print()
print("Completed task-only adapter training.")

print()
print("All poster files were exported to:")
print(RESULTS_EXPORT_DIR.resolve())
print("Vector formats: SVG and PDF")
print("Raster fallback: 300-dpi PNG")
print("Numerical data: CSV")


In [ ]:
# ============================================================
# ACTIVE-TIME-BIN ACCURACY AND CONFUSION MATRICES
#
# Run this cell after training restores the best adapter.
#
# Complete-trial accuracy remains the primary reported metric.
# This diagnostic additionally evaluates every active-valid
# output bin independently.
# ============================================================

required_names = [
    "model",
    "train_loader",
    "val_loader",
    "test_loader",
    "AO_KEPT_CLASSES",
    "AO_CLASS_NAMES",
    "RESULTS_EXPORT_DIR",
    "ao_unpack_batch",
    "map_meta_outputs_to_utah",
    "align_target_and_mask_to_logits",
    "set_model_mode",
    "save_figure_adobe",
]

missing_names = [
    name
    for name in required_names
    if name not in globals()
]

if missing_names:
    raise RuntimeError(
        "Run the earlier setup and training cells first. "
        f"Missing names: {missing_names}"
    )


@torch.no_grad()
def evaluate_active_bins(loader, split_name):
    """
    Evaluate one forced-choice gesture prediction per active-valid time bin.

    The five mapped Meta logits are compared with argmax at each active-valid
    bin. Rest and invalid/padded bins are excluded.
    """
    set_model_mode(False)

    confusion = torch.zeros(
        (AO_KEPT_CLASSES, AO_KEPT_CLASSES),
        dtype=torch.long,
        device="cpu",
    )

    correct_bins = 0
    total_bins = 0

    for batch in loader:
        emg, target, valid_mask, gestures, trial_nums = ao_unpack_batch(batch)

        raw_meta_output = model(emg)
        logits = map_meta_outputs_to_utah(raw_meta_output)

        target, valid_mask = align_target_and_mask_to_logits(
            target,
            valid_mask,
            logits.shape[-1],
        )

        active_mask = target.max(dim=1).values > 0.5
        valid_mask_bool = valid_mask > 0.5
        keep_mask = active_mask & valid_mask_bool

        if not bool(keep_mask.any()):
            continue

        predicted_classes = logits.argmax(dim=1)
        true_classes = target.argmax(dim=1)

        kept_predictions = predicted_classes[keep_mask]
        kept_targets = true_classes[keep_mask]

        correct_bins += int(
            (kept_predictions == kept_targets).sum().item()
        )
        total_bins += int(keep_mask.sum().item())

        flat_confusion_indices = (
            kept_targets * AO_KEPT_CLASSES
            + kept_predictions
        )

        batch_confusion = torch.bincount(
            flat_confusion_indices,
            minlength=AO_KEPT_CLASSES ** 2,
        ).reshape(
            AO_KEPT_CLASSES,
            AO_KEPT_CLASSES,
        )

        confusion += batch_confusion.detach().cpu()

    if total_bins == 0:
        raise RuntimeError(
            f"{split_name} contained no active-valid output bins."
        )

    overall_accuracy = correct_bins / total_bins

    row_totals = confusion.sum(dim=1)
    per_class_accuracy = torch.full(
        (AO_KEPT_CLASSES,),
        float("nan"),
        dtype=torch.float64,
    )

    classes_present = row_totals > 0

    per_class_accuracy[classes_present] = (
        confusion.diag()[classes_present].double()
        / row_totals[classes_present].double()
    )

    balanced_accuracy = float(
        per_class_accuracy[classes_present].mean().item()
    )

    print()
    print("=" * 72)
    print(f"{split_name.upper()} ACTIVE-TIME-BIN RESULTS")
    print("=" * 72)
    print(
        f"Overall active-bin accuracy: "
        f"{correct_bins}/{total_bins} = {overall_accuracy:.4f}"
    )
    print(
        f"Balanced active-bin accuracy: "
        f"{balanced_accuracy:.4f}"
    )

    for class_index, class_name in enumerate(AO_CLASS_NAMES):
        class_total = int(row_totals[class_index].item())
        class_correct = int(
            confusion[class_index, class_index].item()
        )

        if class_total > 0:
            class_accuracy = (
                class_correct / class_total
            )

            print(
                f"  {class_name:<14} "
                f"{class_correct:>4}/{class_total:<4} "
                f"= {class_accuracy:.4f}"
            )
        else:
            print(
                f"  {class_name:<14} "
                "no active-valid bins"
            )

    return {
        "split": split_name,
        "confusion": confusion,
        "correct_bins": correct_bins,
        "total_bins": total_bins,
        "overall_accuracy": overall_accuracy,
        "balanced_accuracy": balanced_accuracy,
        "per_class_accuracy": per_class_accuracy,
    }


def plot_active_bin_confusion(
    metrics,
    output_dir,
    export_stem,
):
    confusion = metrics["confusion"]
    matrix = confusion.numpy()

    figure, axis = plt.subplots(
        figsize=(7.5, 6.5)
    )

    image = axis.imshow(
        matrix,
        interpolation="nearest",
        cmap="Blues",
    )

    figure.colorbar(
        image,
        ax=axis,
        label="Active valid time bins",
    )

    axis.set(
        title=(
            f"{metrics['split']} active-time-bin confusion matrix\n"
            f"Accuracy = {metrics['overall_accuracy']:.1%}"
        ),
        xlabel="Predicted gesture",
        ylabel="True gesture",
        xticks=np.arange(AO_KEPT_CLASSES),
        yticks=np.arange(AO_KEPT_CLASSES),
        xticklabels=AO_CLASS_NAMES,
        yticklabels=AO_CLASS_NAMES,
    )

    plt.setp(
        axis.get_xticklabels(),
        rotation=35,
        ha="right",
        rotation_mode="anchor",
    )

    threshold = (
        matrix.max() / 2.0
        if matrix.size and matrix.max() > 0
        else 0.0
    )

    for true_index in range(AO_KEPT_CLASSES):
        for predicted_index in range(AO_KEPT_CLASSES):
            value = int(
                matrix[true_index, predicted_index]
            )

            axis.text(
                predicted_index,
                true_index,
                str(value),
                ha="center",
                va="center",
                color=(
                    "white"
                    if value > threshold
                    else "black"
                ),
            )

    figure.tight_layout()

    save_figure_adobe(
        figure,
        output_dir,
        export_stem,
    )

    plt.show()
    plt.close(figure)


active_bin_results = {}

for split_key, split_name, split_loader in (
    ("train", "Train", train_loader),
    ("validation", "Validation", val_loader),
    ("test", "Test", test_loader),
):
    metrics = evaluate_active_bins(
        split_loader,
        split_name,
    )

    active_bin_results[split_key] = metrics

    plot_active_bin_confusion(
        metrics,
        output_dir=RESULTS_EXPORT_DIR,
        export_stem=(
            f"{split_key}_active_bin_confusion_matrix"
        ),
    )


print()
print("=" * 72)
print("ACTIVE-TIME-BIN ACCURACY SUMMARY")
print("=" * 72)

for split_key in (
    "train",
    "validation",
    "test",
):
    metrics = active_bin_results[split_key]

    print(
        f"{metrics['split']:<12} "
        f"overall={metrics['overall_accuracy']:.4f} | "
        f"balanced={metrics['balanced_accuracy']:.4f} | "
        f"bins={metrics['total_bins']}"
    )

print()
print(
    "Complete-trial accuracy and active-bin accuracy measure different "
    "behavior and should be labeled separately in figures and text."
)

In [ ]:
# ============================================================
# SIMPLE POSTER RAW-EMG + ACTIVE-BIN CORRECTNESS FIGURES
#
# Run after training restores the best adapter.
#
# Figure design:
#   - actual raw 30 kHz EMG from one channel
#   - x-axis restricted to the active-valid interval
#   - pale green = correct bin
#   - pale red = incorrect bin
#   - small check/X beneath each active bin
#   - no legend, prediction strip, RMS envelope, or correlation
#   - editable SVG + vector PDF + 300-dpi PNG
# ============================================================

from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import torch


# ------------------------------------------------------------
# User controls
# ------------------------------------------------------------

PLOT_SPLITS = (
    "validation",
    "test",
)

# None plots every trial in each selected split.
MAX_TRIALS_PER_SPLIT = None

# Choose the raw Utah channel displayed in every figure:
#
#   0 through 31:
#       Use that same fixed channel in every figure. This is preferred when
#       comparing trials or matching another poster figure.
#
#   None:
#       Automatically select, separately for each trial, the channel with the
#       largest RMS during the active interval.
#
RAW_CHANNEL_INDEX = None

# Separate directory so these do not overwrite the older timeline figures.
SIMPLE_TIMELINE_EXPORT_DIR = (
    Path(RESULTS_EXPORT_DIR)
    / "simple_raw_emg_active_bin_timelines"
)

CORRECT_SHADE_COLOR = "#66BB6A"
INCORRECT_SHADE_COLOR = "#EF5350"

CORRECT_MARK_COLOR = "#1B5E20"
INCORRECT_MARK_COLOR = "#B71C1C"


# ------------------------------------------------------------
# Verify required notebook state
# ------------------------------------------------------------

required_names = [
    "model",
    "train_loader",
    "val_loader",
    "test_loader",
    "AO_KEPT_CLASSES",
    "AO_CLASS_NAMES",
    "META_SAMPLING_RATE",
    "YOUR_SAMPLING_RATE",
    "META_LEFT_CONTEXT",
    "META_OUTPUT_STRIDE",
    "ao_unpack_batch",
    "map_meta_outputs_to_utah",
    "align_target_and_mask_to_logits",
    "extract_one_second_active_centered_trial",
    "set_model_mode",
]

missing_names = [
    name
    for name in required_names
    if name not in globals()
]

if missing_names:
    raise RuntimeError(
        "Run the earlier setup and training cells first. "
        f"Missing names: {missing_names}"
    )


split_loaders = {
    "train": train_loader,
    "validation": val_loader,
    "test": test_loader,
}

unknown_splits = [
    split_name
    for split_name in PLOT_SPLITS
    if split_name not in split_loaders
]

if unknown_splits:
    raise ValueError(
        f"Unknown PLOT_SPLITS entries: {unknown_splits}"
    )

if (
    RAW_CHANNEL_INDEX is not None
    and not 0 <= RAW_CHANNEL_INDEX < 32
):
    raise ValueError(
        "RAW_CHANNEL_INDEX must be None or an integer from 0 to 31."
    )


# ------------------------------------------------------------
# Adobe-compatible saving
# ------------------------------------------------------------

def save_simple_poster_figure(
    figure,
    output_dir,
    stem,
):
    """
    Save editable SVG, vector PDF, and a 300-dpi PNG preview.
    """
    output_dir = Path(output_dir)

    output_dir.mkdir(
        parents=True,
        exist_ok=True,
    )

    saved_paths = []

    # Keep text editable in Illustrator and embed TrueType fonts in PDF.
    with plt.rc_context({
        "svg.fonttype": "none",
        "pdf.fonttype": 42,
        "ps.fonttype": 42,
        "path.simplify": True,
        "path.simplify_threshold": 0.10,
    }):
        for extension, extra_kwargs in (
            ("svg", {}),
            ("pdf", {}),
            ("png", {"dpi": 300}),
        ):
            path = output_dir / f"{stem}.{extension}"

            figure.savefig(
                path,
                bbox_inches="tight",
                pad_inches=0.04,
                facecolor="white",
                transparent=False,
                **extra_kwargs,
            )

            saved_paths.append(path)

    print(f"Saved simplified poster figure: {stem}")

    for path in saved_paths:
        print(" ", path.resolve())

    return saved_paths


# ------------------------------------------------------------
# Recover raw Utah data
# ------------------------------------------------------------

def find_saved_raw_trial(
    loader,
    gesture,
    trial_num,
):
    """
    Locate the original saved trial using its gesture and trial number.
    """
    matches = []

    for trial in loader.dataset.trials:
        saved_gesture = int(
            trial.get("gesture", -1)
        )

        saved_trial_num = int(
            trial.get("trial_num", -1)
        )

        if (
            saved_gesture == gesture
            and saved_trial_num == trial_num
        ):
            matches.append(trial)

    if len(matches) != 1:
        raise RuntimeError(
            "Expected exactly one saved trial for "
            f"G{gesture} trial {trial_num}; "
            f"found {len(matches)}."
        )

    return matches[0]


def recover_raw_one_second_emg(
    loader,
    gesture,
    trial_num,
):
    """
    Recover the one-second, 30 kHz Utah EMG crop before resampling,
    filtering, normalization, or adapter processing.

    Returns [32, 30000].
    """
    saved_trial = find_saved_raw_trial(
        loader,
        gesture,
        trial_num,
    )

    extracted = extract_one_second_active_centered_trial(
        saved_trial
    )

    if extracted is None:
        raise RuntimeError(
            f"G{gesture} trial {trial_num} has no "
            "recoverable active-centered crop."
        )

    raw_emg_time_channels = extracted[0]

    raw_emg_channels_time = (
        raw_emg_time_channels
        .transpose(0, 1)
        .detach()
        .cpu()
        .numpy()
        .astype(np.float64, copy=False)
    )

    if raw_emg_channels_time.shape != (32, 30000):
        raise RuntimeError(
            "Unexpected raw EMG shape: "
            f"{raw_emg_channels_time.shape}"
        )

    return raw_emg_channels_time


# ------------------------------------------------------------
# Raw-channel selection
# ------------------------------------------------------------

def select_raw_emg_channel(
    raw_emg,
    active_start_seconds,
    active_end_seconds,
):
    """
    Use RAW_CHANNEL_INDEX when provided.

    Otherwise select the channel with the largest centered RMS during the
    displayed active interval.
    """
    if RAW_CHANNEL_INDEX is not None:
        return int(RAW_CHANNEL_INDEX)

    start_sample = max(
        0,
        int(
            np.floor(
                active_start_seconds
                * YOUR_SAMPLING_RATE
            )
        ),
    )

    end_sample = min(
        raw_emg.shape[-1],
        int(
            np.ceil(
                active_end_seconds
                * YOUR_SAMPLING_RATE
            )
        ),
    )

    if end_sample <= start_sample:
        raise RuntimeError(
            "The active interval contains no raw EMG samples."
        )

    active_emg = raw_emg[
        :,
        start_sample:end_sample,
    ]

    # Removing a constant channel offset does not alter waveform morphology.
    centered_active_emg = (
        active_emg
        - np.median(
            active_emg,
            axis=1,
            keepdims=True,
        )
    )

    channel_rms = np.sqrt(
        np.mean(
            centered_active_emg ** 2,
            axis=1,
        )
    )

    return int(
        np.argmax(channel_rms)
    )


# ------------------------------------------------------------
# Plot one trial
# ------------------------------------------------------------

def plot_simple_raw_emg_active_bins(
    loader,
    split_name,
    gesture,
    trial_num,
    target,
    valid_mask,
    logits,
):
    """
    Plot one raw EMG channel only over the active-valid interval.

    Correctness is represented by pale green/red bin shading and small
    check/X symbols.
    """
    output_length = int(
        logits.shape[-1]
    )

    if output_length != 198:
        raise RuntimeError(
            "Expected 198 Meta outputs for a one-second 2 kHz input; "
            f"received {output_length}."
        )

    predicted_classes = (
        logits.argmax(dim=0)
        .detach()
        .cpu()
        .numpy()
    )

    true_classes = (
        target.argmax(dim=0)
        .detach()
        .cpu()
        .numpy()
    )

    active_bins = (
        target.max(dim=0).values > 0.5
    ).detach().cpu().numpy()

    valid_bins = (
        valid_mask > 0.5
    ).detach().cpu().numpy()

    scored_bins = (
        active_bins
        & valid_bins
    )

    active_indices = np.flatnonzero(
        scored_bins
    )

    if active_indices.size == 0:
        raise RuntimeError(
            f"{split_name} G{gesture} trial {trial_num} "
            "contains no active-valid output bins."
        )

    correct_bins = (
        predicted_classes
        == true_classes
    )

    correct_active_bins = int(
        correct_bins[active_indices].sum()
    )

    total_active_bins = int(
        active_indices.size
    )

    active_bin_accuracy = (
        correct_active_bins
        / total_active_bins
    )

    # Exact Meta output-center timing.
    output_sample_indices = (
        META_LEFT_CONTEXT
        + np.arange(output_length)
        * META_OUTPUT_STRIDE
    )

    output_times_seconds = (
        output_sample_indices
        / float(META_SAMPLING_RATE)
    )

    half_bin_seconds = (
        META_OUTPUT_STRIDE
        / (
            2.0
            * META_SAMPLING_RATE
        )
    )

    first_active_index = int(
        active_indices[0]
    )

    last_active_index = int(
        active_indices[-1]
    )

    active_start_seconds = max(
        0.0,
        output_times_seconds[first_active_index]
        - half_bin_seconds,
    )

    active_end_seconds = min(
        1.0,
        output_times_seconds[last_active_index]
        + half_bin_seconds,
    )

    raw_emg = recover_raw_one_second_emg(
        loader,
        gesture,
        trial_num,
    )

    raw_channel_index = select_raw_emg_channel(
        raw_emg,
        active_start_seconds,
        active_end_seconds,
    )

    raw_channel = raw_emg[
        raw_channel_index
    ].copy()

    # Remove only the constant vertical offset. No scaling or normalization.
    displayed_start_sample = max(
        0,
        int(
            np.floor(
                active_start_seconds
                * YOUR_SAMPLING_RATE
            )
        ),
    )

    displayed_end_sample = min(
        raw_channel.size,
        int(
            np.ceil(
                active_end_seconds
                * YOUR_SAMPLING_RATE
            )
        ),
    )

    raw_channel -= np.median(
        raw_channel[
            displayed_start_sample:
            displayed_end_sample
        ]
    )

    raw_time_seconds = (
        np.arange(raw_channel.size)
        / float(YOUR_SAMPLING_RATE)
    )

    display_mask = (
        (raw_time_seconds >= active_start_seconds)
        & (raw_time_seconds <= active_end_seconds)
    )

    displayed_time = raw_time_seconds[
        display_mask
    ]

    displayed_signal = raw_channel[
        display_mask
    ]

    if displayed_signal.size == 0:
        raise RuntimeError(
            "No raw EMG samples fall within the active interval."
        )

    # Complete-trial prediction still uses mean logits over active-valid bins.
    scored_bins_tensor = torch.as_tensor(
        scored_bins,
        device=logits.device,
        dtype=torch.bool,
    )

    mean_active_logits = logits[
        :,
        scored_bins_tensor,
    ].mean(dim=1)

    complete_trial_prediction = int(
        mean_active_logits.argmax().item()
    )

    complete_trial_correct = (
        complete_trial_prediction
        == gesture
    )

    trial_status = (
        "correct"
        if complete_trial_correct
        else "incorrect"
    )

    # --------------------------------------------------------
    # Create simplified poster plot
    # --------------------------------------------------------

    with plt.rc_context({
        "font.family": "Arial",
        "font.size": 9,
        "axes.titlesize": 10,
        "axes.labelsize": 9,
        "xtick.labelsize": 8,
        "ytick.labelsize": 8,
        "svg.fonttype": "none",
        "pdf.fonttype": 42,
        "ps.fonttype": 42,
        "path.simplify": True,
        "path.simplify_threshold": 0.10,
    }):
        figure, axis = plt.subplots(
            figsize=(10.0, 3.2)
        )

        # Draw correctness shading behind the raw waveform.
        for bin_index in active_indices:
            bin_time = output_times_seconds[
                bin_index
            ]

            bin_start = max(
                active_start_seconds,
                bin_time
                - half_bin_seconds,
            )

            bin_end = min(
                active_end_seconds,
                bin_time
                + half_bin_seconds,
            )

            shade_color = (
                CORRECT_SHADE_COLOR
                if correct_bins[bin_index]
                else INCORRECT_SHADE_COLOR
            )

            axis.axvspan(
                bin_start,
                bin_end,
                color=shade_color,
                alpha=0.16,
                linewidth=0,
                zorder=0,
            )

        # Actual raw 30 kHz waveform.
        axis.plot(
            displayed_time,
            displayed_signal,
            color="black",
            linewidth=0.50,
            zorder=2,
        )

        signal_min = float(
            np.min(displayed_signal)
        )

        signal_max = float(
            np.max(displayed_signal)
        )

        signal_range = (
            signal_max
            - signal_min
        )

        if (
            not np.isfinite(signal_range)
            or signal_range <= 0
        ):
            signal_range = 1.0

        symbol_y = (
            signal_min
            - 0.10
            * signal_range
        )

        # Small correctness symbols replace the old circles and ticks.
        for bin_index in active_indices:
            is_correct = bool(
                correct_bins[bin_index]
            )

            axis.text(
                output_times_seconds[bin_index],
                symbol_y,
                "✓" if is_correct else "×",
                color=(
                    CORRECT_MARK_COLOR
                    if is_correct
                    else INCORRECT_MARK_COLOR
                ),
                fontsize=5.5,
                fontweight="bold",
                ha="center",
                va="center",
                clip_on=False,
                zorder=3,
            )

        axis.set_xlim(
            active_start_seconds,
            active_end_seconds,
        )

        axis.set_ylim(
            signal_min
            - 0.17 * signal_range,
            signal_max
            + 0.06 * signal_range,
        )

        axis.set_xlabel(
            "Time within one-second trial (s)"
        )

        axis.set_ylabel(
            (
                "Raw EMG amplitude (a.u.)\n"
                f"channel {raw_channel_index}"
            )
        )

        # Requested compact title. Lowercase "complete-trial".
        axis.set_title(
            (
                f"{split_name} — G{gesture} trial {trial_num}: "
                f"{AO_CLASS_NAMES[gesture]}\n"
                f"complete-trial prediction: "
                f"{AO_CLASS_NAMES[complete_trial_prediction]} "
                f"({trial_status}) | "
                f"active-bin accuracy: "
                f"{correct_active_bins}/{total_active_bins} "
                f"({active_bin_accuracy:.1%})"
            ),
            pad=8,
        )

        axis.spines["top"].set_visible(
            False
        )

        axis.spines["right"].set_visible(
            False
        )

        axis.grid(
            False
        )

        # Keep raw acquisition units visible without rescaling the signal.
        axis.ticklabel_format(
            axis="y",
            style="sci",
            scilimits=(-2, 3),
            useMathText=True,
        )

        figure.tight_layout()

        export_stem = (
            f"{split_name.lower()}_"
            f"g{gesture}_"
            f"trial_{trial_num}_"
            "simple_raw_emg_active_bins"
        )

        saved_paths = save_simple_poster_figure(
            figure,
            SIMPLE_TIMELINE_EXPORT_DIR,
            export_stem,
        )

        plt.show()
        plt.close(figure)

    return {
        "split": split_name,
        "gesture": gesture,
        "trial_num": trial_num,
        "raw_channel": raw_channel_index,
        "correct_active_bins": correct_active_bins,
        "total_active_bins": total_active_bins,
        "active_bin_accuracy": active_bin_accuracy,
        "complete_trial_prediction": complete_trial_prediction,
        "complete_trial_correct": complete_trial_correct,
        "saved_paths": saved_paths,
    }


# ------------------------------------------------------------
# Generate validation/test figures
# ------------------------------------------------------------

set_model_mode(False)

simple_timeline_results = []

for split_key in PLOT_SPLITS:
    loader = split_loaders[
        split_key
    ]

    split_name = split_key.capitalize()

    plotted_trials = 0

    print()
    print("=" * 72)
    print(
        f"GENERATING {split_name.upper()} "
        "SIMPLE RAW-EMG FIGURES"
    )
    print("=" * 72)

    with torch.no_grad():
        for batch in loader:
            (
                emg,
                target,
                valid_mask,
                gestures,
                trial_nums,
            ) = ao_unpack_batch(batch)

            raw_meta_output = model(
                emg
            )

            logits = map_meta_outputs_to_utah(
                raw_meta_output
            )

            (
                aligned_target,
                aligned_valid_mask,
            ) = align_target_and_mask_to_logits(
                target,
                valid_mask,
                logits.shape[-1],
            )

            for batch_index in range(
                emg.shape[0]
            ):
                if (
                    MAX_TRIALS_PER_SPLIT
                    is not None
                    and plotted_trials
                    >= MAX_TRIALS_PER_SPLIT
                ):
                    break

                gesture = int(
                    gestures[
                        batch_index
                    ].item()
                )

                trial_num = int(
                    trial_nums[
                        batch_index
                    ].item()
                )

                result = (
                    plot_simple_raw_emg_active_bins(
                        loader=loader,
                        split_name=split_name,
                        gesture=gesture,
                        trial_num=trial_num,
                        target=aligned_target[
                            batch_index
                        ],
                        valid_mask=aligned_valid_mask[
                            batch_index
                        ],
                        logits=logits[
                            batch_index
                        ],
                    )
                )

                simple_timeline_results.append(
                    result
                )

                plotted_trials += 1

            if (
                MAX_TRIALS_PER_SPLIT
                is not None
                and plotted_trials
                >= MAX_TRIALS_PER_SPLIT
            ):
                break


# ------------------------------------------------------------
# Compact output summary
# ------------------------------------------------------------

print()
print("=" * 72)
print("SIMPLE RAW-EMG FIGURE SUMMARY")
print("=" * 72)

for result in simple_timeline_results:
    print(
        f"{result['split']:<10} | "
        f"G{result['gesture']} "
        f"trial {result['trial_num']:>3} | "
        f"raw channel={result['raw_channel']:>2} | "
        f"active-bin accuracy="
        f"{result['correct_active_bins']}/"
        f"{result['total_active_bins']} "
        f"({result['active_bin_accuracy']:.1%})"
    )

print()
print(
    "Saved figures:"
)
print(
    SIMPLE_TIMELINE_EXPORT_DIR.resolve()
)

In [ ]:
# ============================================================
# STANDALONE COMPARISON — RAW UTAH EMG -> TCN -> LSTM -> GESTURE
# ============================================================

import csv
import copy
import random

import numpy as np
import torch
import torch.nn.functional as F
from torch import nn
from torch.utils.data import DataLoader


# The comparison intentionally reuses the exact same preprocessed Dataset
# objects and fixed splits created above. The Meta-TL run is unchanged.
STANDALONE_EPOCHS = 150
STANDALONE_LR = 0.001
STANDALONE_WEIGHT_DECAY = AO_WEIGHT_DECAY
STANDALONE_GRAD_CLIP_NORM = AO_GRAD_CLIP_NORM
STANDALONE_TCN_CHANNELS = (32, 64, 64)
STANDALONE_KERNEL_SIZE = 5
STANDALONE_DROPOUT = 0.20
STANDALONE_LSTM_HIDDEN = 64


def standalone_seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


standalone_seed_everything(SEED)


# A fresh seeded loader makes the standalone run independent of the generator
# state consumed by the Meta-TL run.
standalone_train_generator = torch.Generator().manual_seed(SEED)
standalone_train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    drop_last=False,
    num_workers=0,
    generator=standalone_train_generator,
)


@torch.no_grad()
def fit_standalone_channel_normalization(dataset, batch_size=BATCH_SIZE):
    """Fit per-channel mean/std using TRAIN EMG only."""
    loader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=False,
        drop_last=False,
        num_workers=0,
    )
    sums = torch.zeros(YOUR_CHANNELS, dtype=torch.float64)
    sums_sq = torch.zeros(YOUR_CHANNELS, dtype=torch.float64)
    count = 0

    for emg, _, _, _, _ in loader:
        emg = emg.to(torch.float64)
        sums += emg.sum(dim=(0, 2))
        sums_sq += emg.square().sum(dim=(0, 2))
        count += emg.shape[0] * emg.shape[2]

    mean = sums / count
    variance = (sums_sq / count - mean.square()).clamp_min(0.0)
    std = variance.sqrt().clamp_min(1e-6)
    return (
        mean.float().view(1, YOUR_CHANNELS, 1),
        std.float().view(1, YOUR_CHANNELS, 1),
    )


class StandaloneResidualBlock(nn.Module):
    """Causal two-convolution residual block adapted from the supplied model."""
    def __init__(self, in_channels, out_channels, kernel_size, dilation):
        super().__init__()
        self.left_pad = (kernel_size - 1) * dilation
        self.conv1 = nn.Conv1d(
            in_channels,
            out_channels,
            kernel_size,
            dilation=dilation,
        )
        self.conv2 = nn.Conv1d(
            out_channels,
            out_channels,
            kernel_size,
            dilation=dilation,
        )
        self.downsample = (
            nn.Conv1d(in_channels, out_channels, kernel_size=1)
            if in_channels != out_channels
            else nn.Identity()
        )

    def forward(self, x):
        residual = self.downsample(x)
        out = self.conv1(F.pad(x, (self.left_pad, 0)))
        out = F.relu(out)
        out = self.conv2(F.pad(out, (self.left_pad, 0)))
        return F.relu(out + residual)


class StandaloneEmgDecoder(nn.Module):
    """
    Train-fitted normalization -> 1x1 spatial mixing -> causal TCN ->
    LSTM sequence -> five temporal gesture logits at the exact Meta output bins.

    Gram-Schmidt selection and pose PCA are deliberately omitted because they
    belong to the supplied 42-output pose-regression task, not this classifier.
    """
    def __init__(self, emg_mean, emg_std):
        super().__init__()
        self.register_buffer("emg_mean", emg_mean.clone().float())
        self.register_buffer("emg_std", emg_std.clone().float())
        self.spatial_mix = nn.Conv1d(YOUR_CHANNELS, YOUR_CHANNELS, 1)

        blocks = []
        in_channels = YOUR_CHANNELS
        for block_index, out_channels in enumerate(STANDALONE_TCN_CHANNELS):
            blocks.append(
                StandaloneResidualBlock(
                    in_channels,
                    out_channels,
                    STANDALONE_KERNEL_SIZE,
                    dilation=2 ** block_index,
                )
            )
            in_channels = out_channels
        self.tcn = nn.Sequential(*blocks)
        self.dropout = nn.Dropout(STANDALONE_DROPOUT)
        self.layer_norm = nn.LayerNorm(
            STANDALONE_TCN_CHANNELS[-1],
            elementwise_affine=False,
        )
        self.lstm = nn.LSTM(
            STANDALONE_TCN_CHANNELS[-1],
            STANDALONE_LSTM_HIDDEN,
            batch_first=True,
        )
        self.classifier = nn.Linear(STANDALONE_LSTM_HIDDEN, AO_KEPT_CLASSES)
        self.register_buffer(
            "output_indices",
            torch.arange(
                META_LEFT_CONTEXT,
                META_INPUT_SAMPLES,
                META_OUTPUT_STRIDE,
                dtype=torch.long,
            ),
        )

    def forward(self, emg):
        x = (emg - self.emg_mean) / self.emg_std
        x = self.spatial_mix(x)
        x = self.tcn(x)
        x = self.dropout(x)
        x = self.layer_norm(x.transpose(1, 2))
        sequence, _ = self.lstm(x)
        selected_sequence = sequence.index_select(1, self.output_indices)
        return self.classifier(selected_sequence).transpose(1, 2).contiguous()


standalone_emg_mean, standalone_emg_std = (
    fit_standalone_channel_normalization(train_dataset)
)
standalone_model = StandaloneEmgDecoder(
    standalone_emg_mean,
    standalone_emg_std,
).to(DEVICE)

standalone_parameter_count = sum(
    parameter.numel()
    for parameter in standalone_model.parameters()
    if parameter.requires_grad
)
meta_adapter_parameter_count = sum(
    parameter.numel()
    for parameter in model.adapter.parameters()
    if parameter.requires_grad
)

print("Standalone trainable parameters:", f"{standalone_parameter_count:,}")
print("Meta-TL adapter trainable parameters:", f"{meta_adapter_parameter_count:,}")
print("Both models use the same fixed trials, EMG preprocessing, batch size,")
print("epoch count, LR schedule, active-bin BCE, trial prediction rule,")
print("and validation selection rule.")


standalone_optimizer = torch.optim.AdamW(
    build_adamw_parameter_groups(
        standalone_model,
        STANDALONE_WEIGHT_DECAY,
    ),
    lr=STANDALONE_LR,
)
def standalone_scheduled_lr(epoch_number):
    # Identical schedule to the Meta-TL adapter.
    return scheduled_lr(epoch_number)


def run_standalone_epoch(loader, train):
    standalone_model.train(train)
    loss_sum = 0.0
    batch_count = 0
    correct = 0
    trial_count = 0

    for emg, target, valid_mask, gestures, _ in loader:
        emg = emg.to(DEVICE, non_blocking=True)
        target = target.to(DEVICE, non_blocking=True)
        valid_mask = valid_mask.to(DEVICE, non_blocking=True)
        gestures = gestures.to(DEVICE, non_blocking=True)

        if train:
            standalone_optimizer.zero_grad(set_to_none=True)

        with torch.set_grad_enabled(train):
            logits = standalone_model(emg)
            target, valid_mask = align_target_and_mask_to_logits(
                target,
                valid_mask,
                logits.shape[-1],
            )
            loss = active_only_multilabel_bce(
                logits,
                target,
                valid_mask,
            )

            if not torch.isfinite(loss):
                raise FloatingPointError("Non-finite standalone loss detected.")

            if train:
                loss.backward()
                gradient_norm = torch.nn.utils.clip_grad_norm_(
                    standalone_model.parameters(),
                    STANDALONE_GRAD_CLIP_NORM,
                )
                if not torch.isfinite(gradient_norm):
                    raise FloatingPointError(
                        "Non-finite standalone gradient norm detected."
                    )
                standalone_optimizer.step()

        batch_results = trial_predictions(
            logits.detach(),
            target.detach(),
            valid_mask.detach(),
            gestures.detach(),
        )
        correct += sum(
            int(result["prediction"] == result["true"])
            for result in batch_results
        )
        trial_count += len(batch_results)
        loss_sum += float(loss.detach().item())
        batch_count += 1

    return {
        "loss": loss_sum / max(1, batch_count),
        "accuracy": correct / max(1, trial_count),
        "num_trials": trial_count,
    }


standalone_history = {
    "learning_rate": [],
    "train_loss": [],
    "val_loss": [],
    "train_accuracy": [],
    "val_accuracy": [],
}
standalone_best_state = None
standalone_best_epoch = None
standalone_best_val_accuracy = float("-inf")
standalone_best_val_loss = float("inf")

print("\nStarting standalone decoder training.")
for epoch in range(1, STANDALONE_EPOCHS + 1):
    current_lr = standalone_scheduled_lr(epoch)
    set_optimizer_lr(standalone_optimizer, current_lr)

    train_metrics = run_standalone_epoch(standalone_train_loader, train=True)
    val_metrics = run_standalone_epoch(val_loader, train=False)

    standalone_history["learning_rate"].append(current_lr)
    standalone_history["train_loss"].append(train_metrics["loss"])
    standalone_history["val_loss"].append(val_metrics["loss"])
    standalone_history["train_accuracy"].append(train_metrics["accuracy"])
    standalone_history["val_accuracy"].append(val_metrics["accuracy"])

    print(
        f"Standalone epoch {epoch:03d}/{STANDALONE_EPOCHS} | "
        f"lr {current_lr:.2e} | "
        f"train BCE {train_metrics['loss']:.4f} | "
        f"val BCE {val_metrics['loss']:.4f} | "
        f"train acc {train_metrics['accuracy']:.3f} | "
        f"val acc {val_metrics['accuracy']:.3f}"
    )

    is_better = (
        val_metrics["accuracy"] > standalone_best_val_accuracy
        or (
            np.isclose(
                val_metrics["accuracy"],
                standalone_best_val_accuracy,
            )
            and val_metrics["loss"] < standalone_best_val_loss
        )
    )
    if is_better:
        standalone_best_epoch = epoch
        standalone_best_val_accuracy = val_metrics["accuracy"]
        standalone_best_val_loss = val_metrics["loss"]
        standalone_best_state = copy.deepcopy(standalone_model.state_dict())


if standalone_best_state is None:
    raise RuntimeError("No standalone best state was recorded.")

standalone_model.load_state_dict(standalone_best_state)
standalone_model.to(DEVICE)
standalone_test_metrics = run_standalone_epoch(test_loader, train=False)

print("\nRestored standalone epoch", standalone_best_epoch)
print("Standalone best validation accuracy:", standalone_best_val_accuracy)
print("Standalone best validation active-bin BCE:", standalone_best_val_loss)
print("Standalone blind test active-bin BCE:", standalone_test_metrics["loss"])
print("Standalone blind test accuracy:", standalone_test_metrics["accuracy"])


In [ ]:
# ============================================================
# DIRECT META-TL VS STANDALONE LEARNING-CURVE COMPARISON
# ============================================================

COMPARISON_EXPORT_DIR = RESULTS_EXPORT_DIR / "meta_vs_standalone"
COMPARISON_EXPORT_DIR.mkdir(parents=True, exist_ok=True)

comparison_epochs = np.arange(1, AO_EPOCHS + 1)
meta_learning_rate = np.asarray(
    [scheduled_lr(epoch) for epoch in comparison_epochs],
    dtype=float,
)

comparison_figure, axes = plt.subplots(
    2,
    2,
    figsize=(12.0, 8.0),
    sharex=True,
)

axes[0, 0].plot(
    comparison_epochs,
    history["train_task_loss"],
    label="Meta-TL active-bin BCE",
    linewidth=2,
)
axes[0, 0].plot(
    comparison_epochs,
    standalone_history["train_loss"],
    label="Standalone active-bin BCE",
    linewidth=2,
)
axes[0, 0].set_title("Training loss")
axes[0, 0].set_ylabel("Loss")

axes[0, 1].plot(
    comparison_epochs,
    history["val_task_loss"],
    label="Meta-TL active-bin BCE",
    linewidth=2,
)
axes[0, 1].plot(
    comparison_epochs,
    standalone_history["val_loss"],
    label="Standalone active-bin BCE",
    linewidth=2,
)
axes[0, 1].set_title("Validation loss")
axes[0, 1].set_ylabel("Loss")

axes[1, 0].plot(
    comparison_epochs,
    100.0 * np.asarray(history["train_accuracy"]),
    label="Meta-TL",
    linewidth=2,
)
axes[1, 0].plot(
    comparison_epochs,
    100.0 * np.asarray(standalone_history["train_accuracy"]),
    label="Standalone",
    linewidth=2,
)
axes[1, 0].set_title("Training trial accuracy")
axes[1, 0].set_ylabel("Accuracy (%)")
axes[1, 0].set_xlabel("Epoch")
axes[1, 0].set_ylim(-2, 102)

axes[1, 1].plot(
    comparison_epochs,
    100.0 * np.asarray(history["val_accuracy"]),
    label="Meta-TL",
    linewidth=2,
)
axes[1, 1].plot(
    comparison_epochs,
    100.0 * np.asarray(standalone_history["val_accuracy"]),
    label="Standalone",
    linewidth=2,
)
axes[1, 1].set_title("Validation trial accuracy")
axes[1, 1].set_ylabel("Accuracy (%)")
axes[1, 1].set_xlabel("Epoch")
axes[1, 1].set_ylim(-2, 102)

for axis in axes.reshape(-1):
    axis.grid(alpha=0.25)
    axis.legend(frameon=False)

comparison_figure.suptitle(
    "Same Utah splits and LR schedule: frozen Meta transfer vs standalone decoder",
    fontsize=13,
)
comparison_figure.tight_layout()
save_figure_adobe(
    comparison_figure,
    COMPARISON_EXPORT_DIR,
    "meta_tl_vs_standalone_learning_curves",
)
plt.show()


lr_figure, lr_axis = plt.subplots(figsize=(8.0, 3.6))
lr_axis.plot(
    comparison_epochs,
    meta_learning_rate,
    color="black",
    linewidth=2,
    label="Both models",
)
lr_axis.set_title("Shared learning-rate schedule")
lr_axis.set_xlabel("Epoch")
lr_axis.set_ylabel("Learning rate")
lr_axis.set_yscale("log")
lr_axis.grid(alpha=0.25)
lr_axis.legend(frameon=False)
lr_figure.tight_layout()
save_figure_adobe(
    lr_figure,
    COMPARISON_EXPORT_DIR,
    "shared_learning_rate_schedule",
)
plt.show()


history_csv_path = COMPARISON_EXPORT_DIR / "meta_tl_vs_standalone_history.csv"
with history_csv_path.open("w", newline="", encoding="utf-8") as csv_file:
    writer = csv.writer(csv_file)
    writer.writerow([
        "epoch",
        "learning_rate",
        "meta_train_bce",
        "meta_val_bce",
        "meta_train_accuracy",
        "meta_val_accuracy",
        "standalone_train_bce",
        "standalone_val_bce",
        "standalone_train_accuracy",
        "standalone_val_accuracy",
    ])
    for index, epoch in enumerate(comparison_epochs):
        writer.writerow([
            int(epoch),
            float(meta_learning_rate[index]),
            float(history["train_task_loss"][index]),
            float(history["val_task_loss"][index]),
            float(history["train_accuracy"][index]),
            float(history["val_accuracy"][index]),
            float(standalone_history["train_loss"][index]),
            float(standalone_history["val_loss"][index]),
            float(standalone_history["train_accuracy"][index]),
            float(standalone_history["val_accuracy"][index]),
        ])

standalone_checkpoint_path = (
    COMPARISON_EXPORT_DIR / "best_standalone_emg_decoder.pt"
)
torch.save(
    {
        "state_dict": {
            key: value.detach().cpu()
            for key, value in standalone_model.state_dict().items()
        },
        "best_epoch": standalone_best_epoch,
        "best_val_accuracy": standalone_best_val_accuracy,
        "best_val_loss": standalone_best_val_loss,
        "test_metrics": standalone_test_metrics,
        "history": standalone_history,
        "class_names": AO_CLASS_NAMES,
        "data_path": str(DATA_PATH),
        "architecture": {
            "kind": "normalized-spatial-mix-causal-tcn-lstm-temporal-classifier",
            "input_channels": YOUR_CHANNELS,
            "input_samples": META_INPUT_SAMPLES,
            "tcn_channels": STANDALONE_TCN_CHANNELS,
            "kernel_size": STANDALONE_KERNEL_SIZE,
            "dropout": STANDALONE_DROPOUT,
            "lstm_hidden": STANDALONE_LSTM_HIDDEN,
            "output_classes": AO_KEPT_CLASSES,
            "output_bins": EXPECTED_META_OUTPUT_SAMPLES,
        },
    },
    standalone_checkpoint_path,
)

print("Comparison history:", history_csv_path.resolve())
print("Standalone checkpoint:", standalone_checkpoint_path.resolve())
print()
print("Both loss curves now use the same active-bin BCE definition.")
print("Both accuracy curves use the same active-bin mean-logit trial decision.")


## Standalone decoder comparison

The original Meta-TL pipeline above is unchanged. The next cell trains a separate decoder on the same fixed trials and preprocessed EMG. Its architecture retains the supplied model's spatial-mix, causal-TCN, and LSTM core, adapted to the same five-class temporal output, active-bin loss, and trial decision rule used by Meta-TL.


In [ ]:
# ============================================================
# STANDALONE COMPARISON — RAW UTAH EMG -> TCN -> LSTM -> GESTURE
# ============================================================

import csv
import copy
import random

import numpy as np
import torch
import torch.nn.functional as F
from torch import nn
from torch.utils.data import DataLoader


# The comparison intentionally reuses the exact same preprocessed Dataset
# objects and fixed splits created above. The Meta-TL run is unchanged.
STANDALONE_EPOCHS = AO_EPOCHS
STANDALONE_LR = AO_LR
STANDALONE_WEIGHT_DECAY = AO_WEIGHT_DECAY
STANDALONE_GRAD_CLIP_NORM = AO_GRAD_CLIP_NORM
STANDALONE_TCN_CHANNELS = (32, 64, 64)
STANDALONE_KERNEL_SIZE = 5
STANDALONE_DROPOUT = 0.20
STANDALONE_LSTM_HIDDEN = 64


def standalone_seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


standalone_seed_everything(SEED)


# A fresh seeded loader makes the standalone run independent of the generator
# state consumed by the Meta-TL run.
standalone_train_generator = torch.Generator().manual_seed(SEED)
standalone_train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    drop_last=False,
    num_workers=0,
    generator=standalone_train_generator,
)


@torch.no_grad()
def fit_standalone_channel_normalization(dataset, batch_size=BATCH_SIZE):
    """Fit per-channel mean/std using TRAIN EMG only."""
    loader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=False,
        drop_last=False,
        num_workers=0,
    )
    sums = torch.zeros(YOUR_CHANNELS, dtype=torch.float64)
    sums_sq = torch.zeros(YOUR_CHANNELS, dtype=torch.float64)
    count = 0

    for emg, _, _, _, _ in loader:
        emg = emg.to(torch.float64)
        sums += emg.sum(dim=(0, 2))
        sums_sq += emg.square().sum(dim=(0, 2))
        count += emg.shape[0] * emg.shape[2]

    mean = sums / count
    variance = (sums_sq / count - mean.square()).clamp_min(0.0)
    std = variance.sqrt().clamp_min(1e-6)
    return (
        mean.float().view(1, YOUR_CHANNELS, 1),
        std.float().view(1, YOUR_CHANNELS, 1),
    )


class StandaloneResidualBlock(nn.Module):
    """Causal two-convolution residual block adapted from the supplied model."""
    def __init__(self, in_channels, out_channels, kernel_size, dilation):
        super().__init__()
        self.left_pad = (kernel_size - 1) * dilation
        self.conv1 = nn.Conv1d(
            in_channels,
            out_channels,
            kernel_size,
            dilation=dilation,
        )
        self.conv2 = nn.Conv1d(
            out_channels,
            out_channels,
            kernel_size,
            dilation=dilation,
        )
        self.downsample = (
            nn.Conv1d(in_channels, out_channels, kernel_size=1)
            if in_channels != out_channels
            else nn.Identity()
        )

    def forward(self, x):
        residual = self.downsample(x)
        out = self.conv1(F.pad(x, (self.left_pad, 0)))
        out = F.relu(out)
        out = self.conv2(F.pad(out, (self.left_pad, 0)))
        return F.relu(out + residual)


class StandaloneEmgDecoder(nn.Module):
    """
    Train-fitted normalization -> 1x1 spatial mixing -> causal TCN ->
    LSTM sequence -> five temporal gesture logits at the exact Meta output bins.

    Gram-Schmidt selection and pose PCA are deliberately omitted because they
    belong to the supplied 42-output pose-regression task, not this classifier.
    """
    def __init__(self, emg_mean, emg_std):
        super().__init__()
        self.register_buffer("emg_mean", emg_mean.clone().float())
        self.register_buffer("emg_std", emg_std.clone().float())
        self.spatial_mix = nn.Conv1d(YOUR_CHANNELS, YOUR_CHANNELS, 1)

        blocks = []
        in_channels = YOUR_CHANNELS
        for block_index, out_channels in enumerate(STANDALONE_TCN_CHANNELS):
            blocks.append(
                StandaloneResidualBlock(
                    in_channels,
                    out_channels,
                    STANDALONE_KERNEL_SIZE,
                    dilation=2 ** block_index,
                )
            )
            in_channels = out_channels
        self.tcn = nn.Sequential(*blocks)
        self.dropout = nn.Dropout(STANDALONE_DROPOUT)
        self.layer_norm = nn.LayerNorm(
            STANDALONE_TCN_CHANNELS[-1],
            elementwise_affine=False,
        )
        self.lstm = nn.LSTM(
            STANDALONE_TCN_CHANNELS[-1],
            STANDALONE_LSTM_HIDDEN,
            batch_first=True,
        )
        self.classifier = nn.Linear(STANDALONE_LSTM_HIDDEN, AO_KEPT_CLASSES)
        self.register_buffer(
            "output_indices",
            torch.arange(
                META_LEFT_CONTEXT,
                META_INPUT_SAMPLES,
                META_OUTPUT_STRIDE,
                dtype=torch.long,
            ),
        )

    def forward(self, emg):
        x = (emg - self.emg_mean) / self.emg_std
        x = self.spatial_mix(x)
        x = self.tcn(x)
        x = self.dropout(x)
        x = self.layer_norm(x.transpose(1, 2))
        sequence, _ = self.lstm(x)
        selected_sequence = sequence.index_select(1, self.output_indices)
        return self.classifier(selected_sequence).transpose(1, 2).contiguous()


standalone_emg_mean, standalone_emg_std = (
    fit_standalone_channel_normalization(train_dataset)
)
standalone_model = StandaloneEmgDecoder(
    standalone_emg_mean,
    standalone_emg_std,
).to(DEVICE)

standalone_parameter_count = sum(
    parameter.numel()
    for parameter in standalone_model.parameters()
    if parameter.requires_grad
)
meta_adapter_parameter_count = sum(
    parameter.numel()
    for parameter in model.adapter.parameters()
    if parameter.requires_grad
)

print("Standalone trainable parameters:", f"{standalone_parameter_count:,}")
print("Meta-TL adapter trainable parameters:", f"{meta_adapter_parameter_count:,}")
print("Both models use the same fixed trials, EMG preprocessing, batch size,")
print("epoch count, LR schedule, active-bin supervision, trial prediction rule,")
print("and validation selection rule.")
print("Base LR: Meta-TL", AO_LR, "| standalone", STANDALONE_LR)


standalone_optimizer = torch.optim.AdamW(
    build_adamw_parameter_groups(
        standalone_model,
        STANDALONE_WEIGHT_DECAY,
    ),
    lr=STANDALONE_LR,
)
def standalone_scheduled_lr(epoch_number):
    # Same warmup/decay shape as Meta-TL, but based on the standalone LR.
    if epoch_number <= AO_WARMUP_EPOCHS:
        fraction = epoch_number / max(1, AO_WARMUP_EPOCHS)
        return STANDALONE_LR * fraction
    if epoch_number >= AO_DECAY_EPOCH:
        return STANDALONE_LR * AO_DECAY_FACTOR
    return STANDALONE_LR


def active_only_multiclass_ce(logits, target, valid_mask):
    """Five-way cross-entropy evaluated only at active valid output bins."""
    active = target.max(dim=1).values > 0.5
    valid = valid_mask > 0.5
    keep = active & valid
    if int(keep.sum().item()) == 0:
        raise RuntimeError("Batch contains no active valid target bins.")

    true_classes = target.argmax(dim=1)
    logits_time_classes = logits.transpose(1, 2).contiguous()
    return F.cross_entropy(
        logits_time_classes[keep],
        true_classes[keep],
    )


def run_standalone_epoch(loader, train):
    standalone_model.train(train)
    loss_sum = 0.0
    batch_count = 0
    correct = 0
    trial_count = 0
    gradient_norm_sum = 0.0
    gradient_step_count = 0

    for emg, target, valid_mask, gestures, _ in loader:
        emg = emg.to(DEVICE, non_blocking=True)
        target = target.to(DEVICE, non_blocking=True)
        valid_mask = valid_mask.to(DEVICE, non_blocking=True)
        gestures = gestures.to(DEVICE, non_blocking=True)

        if train:
            standalone_optimizer.zero_grad(set_to_none=True)

        with torch.set_grad_enabled(train):
            logits = standalone_model(emg)
            target, valid_mask = align_target_and_mask_to_logits(
                target,
                valid_mask,
                logits.shape[-1],
            )
            loss = active_only_multiclass_ce(
                logits,
                target,
                valid_mask,
            )

            if not torch.isfinite(loss):
                raise FloatingPointError("Non-finite standalone loss detected.")

            if train:
                loss.backward()
                gradient_norm = torch.nn.utils.clip_grad_norm_(
                    standalone_model.parameters(),
                    STANDALONE_GRAD_CLIP_NORM,
                )
                if not torch.isfinite(gradient_norm):
                    raise FloatingPointError(
                        "Non-finite standalone gradient norm detected."
                    )
                if float(gradient_norm.detach().item()) <= 0.0:
                    raise RuntimeError(
                        "Standalone gradient norm is zero; backpropagation is disconnected."
                    )
                standalone_optimizer.step()
                gradient_norm_sum += float(gradient_norm.detach().item())
                gradient_step_count += 1

        batch_results = trial_predictions(
            logits.detach(),
            target.detach(),
            valid_mask.detach(),
            gestures.detach(),
        )
        correct += sum(
            int(result["prediction"] == result["true"])
            for result in batch_results
        )
        trial_count += len(batch_results)
        loss_sum += float(loss.detach().item())
        batch_count += 1

    return {
        "loss": loss_sum / max(1, batch_count),
        "accuracy": correct / max(1, trial_count),
        "num_trials": trial_count,
        "gradient_norm": (
            gradient_norm_sum / gradient_step_count
            if gradient_step_count
            else float("nan")
        ),
    }


standalone_history = {
    "learning_rate": [],
    "train_loss": [],
    "val_loss": [],
    "train_accuracy": [],
    "val_accuracy": [],
    "train_gradient_norm": [],
}
standalone_best_state = None
standalone_best_epoch = None
standalone_best_val_accuracy = float("-inf")
standalone_best_val_loss = float("inf")

print("\nStarting standalone decoder training.")
for epoch in range(1, STANDALONE_EPOCHS + 1):
    current_lr = standalone_scheduled_lr(epoch)
    set_optimizer_lr(standalone_optimizer, current_lr)

    train_metrics = run_standalone_epoch(standalone_train_loader, train=True)
    val_metrics = run_standalone_epoch(val_loader, train=False)

    standalone_history["learning_rate"].append(current_lr)
    standalone_history["train_loss"].append(train_metrics["loss"])
    standalone_history["val_loss"].append(val_metrics["loss"])
    standalone_history["train_accuracy"].append(train_metrics["accuracy"])
    standalone_history["val_accuracy"].append(val_metrics["accuracy"])
    standalone_history["train_gradient_norm"].append(
        train_metrics["gradient_norm"]
    )

    print(
        f"Standalone epoch {epoch:03d}/{STANDALONE_EPOCHS} | "
        f"lr {current_lr:.2e} | "
        f"train CE {train_metrics['loss']:.4f} | "
        f"val CE {val_metrics['loss']:.4f} | "
        f"train acc {train_metrics['accuracy']:.3f} | "
        f"val acc {val_metrics['accuracy']:.3f} | "
        f"grad {train_metrics['gradient_norm']:.3e}"
    )

    is_better = (
        val_metrics["accuracy"] > standalone_best_val_accuracy
        or (
            np.isclose(
                val_metrics["accuracy"],
                standalone_best_val_accuracy,
            )
            and val_metrics["loss"] < standalone_best_val_loss
        )
    )
    if is_better:
        standalone_best_epoch = epoch
        standalone_best_val_accuracy = val_metrics["accuracy"]
        standalone_best_val_loss = val_metrics["loss"]
        standalone_best_state = copy.deepcopy(standalone_model.state_dict())


if standalone_best_state is None:
    raise RuntimeError("No standalone best state was recorded.")

standalone_model.load_state_dict(standalone_best_state)
standalone_model.to(DEVICE)
standalone_test_metrics = run_standalone_epoch(test_loader, train=False)

print("\nRestored standalone epoch", standalone_best_epoch)
print("Standalone best validation accuracy:", standalone_best_val_accuracy)
print("Standalone best validation active-bin CE:", standalone_best_val_loss)
print("Standalone blind test active-bin CE:", standalone_test_metrics["loss"])
print("Standalone blind test accuracy:", standalone_test_metrics["accuracy"])


In [ ]:
# ============================================================
# DIRECT META-TL VS STANDALONE LEARNING-CURVE COMPARISON
# ============================================================

COMPARISON_EXPORT_DIR = RESULTS_EXPORT_DIR / "meta_vs_standalone"
COMPARISON_EXPORT_DIR.mkdir(parents=True, exist_ok=True)

comparison_epochs = np.arange(1, AO_EPOCHS + 1)
meta_learning_rate = np.asarray(
    [scheduled_lr(epoch) for epoch in comparison_epochs],
    dtype=float,
)

comparison_figure, axes = plt.subplots(
    2,
    2,
    figsize=(12.0, 8.0),
    sharex=True,
)

axes[0, 0].plot(
    comparison_epochs,
    history["train_task_loss"],
    label="Train",
    linewidth=2,
)
axes[0, 0].plot(
    comparison_epochs,
    history["val_task_loss"],
    label="Validation",
    linewidth=2,
)
axes[0, 0].set_title("Meta-TL active-bin BCE")
axes[0, 0].set_ylabel("Loss")

axes[0, 1].plot(
    comparison_epochs,
    standalone_history["train_loss"],
    label="Train",
    linewidth=2,
)
axes[0, 1].plot(
    comparison_epochs,
    standalone_history["val_loss"],
    label="Validation",
    linewidth=2,
)
axes[0, 1].set_title("Standalone active-bin multiclass CE")
axes[0, 1].set_ylabel("Loss")

axes[1, 0].plot(
    comparison_epochs,
    100.0 * np.asarray(history["train_accuracy"]),
    label="Meta-TL",
    linewidth=2,
)
axes[1, 0].plot(
    comparison_epochs,
    100.0 * np.asarray(standalone_history["train_accuracy"]),
    label="Standalone",
    linewidth=2,
)
axes[1, 0].set_title("Training trial accuracy")
axes[1, 0].set_ylabel("Accuracy (%)")
axes[1, 0].set_xlabel("Epoch")
axes[1, 0].set_ylim(-2, 102)

axes[1, 1].plot(
    comparison_epochs,
    100.0 * np.asarray(history["val_accuracy"]),
    label="Meta-TL",
    linewidth=2,
)
axes[1, 1].plot(
    comparison_epochs,
    100.0 * np.asarray(standalone_history["val_accuracy"]),
    label="Standalone",
    linewidth=2,
)
axes[1, 1].set_title("Validation trial accuracy")
axes[1, 1].set_ylabel("Accuracy (%)")
axes[1, 1].set_xlabel("Epoch")
axes[1, 1].set_ylim(-2, 102)

for axis in axes.reshape(-1):
    axis.grid(alpha=0.25)
    axis.legend(frameon=False)

comparison_figure.suptitle(
    "Matched data and evaluation: frozen Meta transfer vs standalone decoder",
    fontsize=13,
)
comparison_figure.tight_layout()
save_figure_adobe(
    comparison_figure,
    COMPARISON_EXPORT_DIR,
    "meta_tl_vs_standalone_learning_curves",
)
plt.show()


lr_figure, lr_axis = plt.subplots(figsize=(8.0, 3.6))
lr_axis.plot(
    comparison_epochs,
    meta_learning_rate,
    color="tab:blue",
    linewidth=2,
    label="Meta-TL",
)
lr_axis.plot(
    comparison_epochs,
    standalone_history["learning_rate"],
    color="tab:orange",
    linewidth=2,
    label="Standalone",
)
lr_axis.set_title("Shared learning-rate schedule")
lr_axis.set_xlabel("Epoch")
lr_axis.set_ylabel("Learning rate")
lr_axis.set_yscale("log")
lr_axis.grid(alpha=0.25)
lr_axis.legend(frameon=False)
lr_figure.tight_layout()
save_figure_adobe(
    lr_figure,
    COMPARISON_EXPORT_DIR,
    "shared_learning_rate_schedule",
)
plt.show()


history_csv_path = COMPARISON_EXPORT_DIR / "meta_tl_vs_standalone_history.csv"
with history_csv_path.open("w", newline="", encoding="utf-8") as csv_file:
    writer = csv.writer(csv_file)
    writer.writerow([
        "epoch",
        "meta_learning_rate",
        "standalone_learning_rate",
        "meta_train_bce",
        "meta_val_bce",
        "meta_train_accuracy",
        "meta_val_accuracy",
        "standalone_train_active_bin_ce",
        "standalone_val_active_bin_ce",
        "standalone_train_accuracy",
        "standalone_val_accuracy",
    ])
    for index, epoch in enumerate(comparison_epochs):
        writer.writerow([
            int(epoch),
            float(meta_learning_rate[index]),
            float(standalone_history["learning_rate"][index]),
            float(history["train_task_loss"][index]),
            float(history["val_task_loss"][index]),
            float(history["train_accuracy"][index]),
            float(history["val_accuracy"][index]),
            float(standalone_history["train_loss"][index]),
            float(standalone_history["val_loss"][index]),
            float(standalone_history["train_accuracy"][index]),
            float(standalone_history["val_accuracy"][index]),
        ])

standalone_checkpoint_path = (
    COMPARISON_EXPORT_DIR / "best_standalone_emg_decoder.pt"
)
torch.save(
    {
        "state_dict": {
            key: value.detach().cpu()
            for key, value in standalone_model.state_dict().items()
        },
        "best_epoch": standalone_best_epoch,
        "best_val_accuracy": standalone_best_val_accuracy,
        "best_val_loss": standalone_best_val_loss,
        "test_metrics": standalone_test_metrics,
        "history": standalone_history,
        "class_names": AO_CLASS_NAMES,
        "data_path": str(DATA_PATH),
        "architecture": {
            "kind": "normalized-spatial-mix-causal-tcn-lstm-temporal-classifier",
            "input_channels": YOUR_CHANNELS,
            "input_samples": META_INPUT_SAMPLES,
            "tcn_channels": STANDALONE_TCN_CHANNELS,
            "kernel_size": STANDALONE_KERNEL_SIZE,
            "dropout": STANDALONE_DROPOUT,
            "lstm_hidden": STANDALONE_LSTM_HIDDEN,
            "output_classes": AO_KEPT_CLASSES,
            "output_bins": EXPECTED_META_OUTPUT_SAMPLES,
        },
    },
    standalone_checkpoint_path,
)

print("Comparison history:", history_csv_path.resolve())
print("Standalone checkpoint:", standalone_checkpoint_path.resolve())
print()
print("Meta-TL loss is active-bin BCE; standalone loss is active-bin multiclass CE.")
print("Losses are plotted separately because their numerical scales differ.")
print("Both accuracy curves use the same active-bin mean-logit trial decision.")
